In [1]:
"""
Functional TIME SERIES simulation + FPCA and FAE + time series analysis
Batch runner: run N=100 reps.

Dependencies:
  numpy, torch, scikit-learn, pandas, openpyxl
"""

from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List

import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd


# ============================================================
# 0) Repro / metrics / utilities  
# ============================================================

def set_seed(seed: int = 123) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def rel_mse(y_hat: torch.Tensor, y_true: torch.Tensor, eps: float = 1e-12) -> float:
    """
    relMSE = MSE / Var(y_true)
    """
    m = torch.mean((y_hat - y_true) ** 2)
    v = torch.var(y_true, unbiased=False)
    return float((m / v.clamp_min(eps)).detach().cpu())

def trapezoid_weights(x: np.ndarray) -> torch.Tensor:
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    if len(dx) < 1:
        raise ValueError("Need at least 2 grid points.")
    w = np.zeros_like(x)
    w[0] = dx[0] / 2.0
    w[-1] = dx[-1] / 2.0
    if len(x) > 2:
        w[1:-1] = (x[2:] - x[:-2]) / 2.0
    return torch.tensor(w, dtype=torch.float32)

# Test and Train Splitting (kepping horizonal curves for forecasting)
def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    if horizon < 1:
        raise ValueError("horizon must be >= 1")
    return X[:-horizon], X[-horizon:]

def print_report_table(title: str, rows: Dict[str, Dict[str, float]]) -> None:
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))
    header = f"{'Method':<14}  {'relMSE vs CLEAN':>16}  {'relMSE vs NOISY':>16}"
    print(header)
    print("-" * len(header))
    for method, vals in rows.items():
        a = vals.get("relMSE_vs_clean", float("nan"))
        b = vals.get("relMSE_vs_noisy", float("nan"))
        print(f"{method:<14}  {a:16.6f}  {b:16.6f}")


# ============================================================
# 1) Basis builder (Fourier / Bspline)  
# ============================================================

class BasisFCBuilder:
    """
    Build basis matrix evaluated on tpts.
    Output shape: [n_time, n_basis]
    """
    def __init__(self, n_basis=20, basis_type="Fourier", custom_basis_fn=None, bspline_degree=3):
        self.n_basis = int(n_basis)
        self.basis_type = basis_type
        self.basis_type_l = basis_type.lower()
        self.custom_basis_fn = custom_basis_fn
        self.bspline_degree = int(bspline_degree)

    def build(self, tpts: torch.Tensor) -> torch.Tensor:
        if self.custom_basis_fn is not None:
            B = self.custom_basis_fn(tpts)
            if not torch.is_tensor(B):
                B = torch.tensor(B, dtype=torch.float32)
            return B.float()

        if self.basis_type_l == "fourier":
            return self._build_fourier(tpts)
        elif self.basis_type_l in ("bspline", "b-spline", "b_spline"):
            return self._build_bspline(tpts, degree=self.bspline_degree)
        else:
            raise ValueError("basis_type must be 'Fourier' or 'Bspline'.")

    def _build_fourier(self, tpts: torch.Tensor) -> torch.Tensor:
        t = tpts.flatten()
        t_min = t.min()
        t_max = t.max()
        denom = (t_max - t_min).clamp_min(1e-8)
        tau = (t - t_min) / denom  # [0,1]

        n_time = tpts.shape[0]
        n_basis = self.n_basis
        device = t.device

        B = torch.zeros(n_time, n_basis, dtype=torch.float32, device=device)
        if n_basis > 0:
            B[:, 0] = 1.0

        k = 1
        idx = 1
        while idx < n_basis:
            B[:, idx] = torch.sin(2.0 * math.pi * k * tau)
            idx += 1
            if idx < n_basis:
                B[:, idx] = torch.cos(2.0 * math.pi * k * tau)
                idx += 1
            k += 1
        return B

    def _build_bspline(self, tpts: torch.Tensor, degree: int = 3) -> torch.Tensor:
        t = tpts.flatten()
        t_min = t.min()
        t_max = t.max()
        denom = (t_max - t_min).clamp_min(1e-8)
        tau = (t - t_min) / denom
        tau_np = tau.detach().cpu().numpy()

        n_time = tau_np.shape[0]
        n_basis = self.n_basis
        p = degree

        if n_basis < p + 1:
            raise ValueError(f"Bspline: n_basis={n_basis} must be at least degree+1={p+1}.")

        n_int = max(n_basis - p - 1, 0)
        if n_int > 0:
            interior = np.linspace(0.0, 1.0, n_int + 2)[1:-1]
            knots = np.concatenate((np.zeros(p + 1), interior, np.ones(p + 1)))
        else:
            knots = np.concatenate((np.zeros(p + 1), np.ones(p + 1)))

        N = np.zeros((n_basis, n_time), dtype=np.float64)
        for i in range(n_basis):
            left = knots[i]
            right = knots[i + 1]
            N[i, :] = np.where((tau_np >= left) & (tau_np < right), 1.0, 0.0)
        N[-1, tau_np == 1.0] = 1.0

        for k in range(1, p + 1):
            N_next = np.zeros_like(N)
            for i in range(n_basis):
                denom_left = knots[i + k] - knots[i]
                if denom_left > 0:
                    coeff_left = (tau_np - knots[i]) / denom_left
                    N_left = coeff_left * N[i, :]
                else:
                    N_left = 0.0

                denom_right = (knots[i + k + 1] - knots[i + 1]) if (i + 1) < n_basis else 0.0
                if denom_right > 0 and (i + 1) < n_basis:
                    coeff_right = (knots[i + k + 1] - tau_np) / denom_right
                    N_right = coeff_right * N[i + 1, :]
                else:
                    N_right = 0.0

                N_next[i, :] = N_left + N_right
            N = N_next

        return torch.tensor(N.T, dtype=torch.float32, device=tpts.device)


# ============================================================
# 2) Simulator  
# ============================================================

# Stationary Condition 
def spectral_radius(A: np.ndarray) -> float:
    return float(np.max(np.abs(np.linalg.eigvals(A))))

def stabilize_A(A: np.ndarray, target_rho: float = 0.9) -> np.ndarray:
    rho = spectral_radius(A)
    if rho <= target_rho or rho <= 1e-12:
        return A
    return (target_rho / rho) * A

def generate_latent_var1(T: int, d: int, Sigma: np.ndarray, seed: int, target_rho: float = 0.8):
    rng = np.random.default_rng(seed)
    A = rng.normal(size=(d, d)) * 0.2
    A = stabilize_A(A, target_rho=target_rho)

    Z = np.zeros((T, d), dtype=float)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma)

    gamma = 0.15
    for t in range(1, T):
        eps = rng.multivariate_normal(np.zeros(d), Sigma)
        Z[t] = A @ Z[t - 1] + gamma * np.tanh(A @ Z[t - 1]) + eps

    return Z, A

class MLPMap(nn.Module):
    def __init__(self, d: int, M: int, hidden=(64, 64), activation="tanh"):
        super().__init__()
        act = {"relu": nn.ReLU(), "gelu": nn.GELU(), "sigmoid": nn.Sigmoid(), "tanh": nn.Tanh()}.get(
            activation.lower(), nn.Tanh()
        )
        layers = []
        in_dim = d
        for h in hidden:
            layers += [nn.Linear(in_dim, h), act]
            in_dim = h
        layers += [nn.Linear(in_dim, M)]
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

class LinearMap(nn.Module):
    def __init__(self, d: int, hidden: list[int], M: int, bias: bool = True, activation=nn.Identity()):
        super().__init__()
        dims = [d] + hidden + [M]

        self.layers = nn.ModuleList(
            [nn.Linear(dims[i], dims[i + 1], bias=bias) for i in range(len(dims) - 1)]
        )
        self.activation = activation

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = z
        for i, layer in enumerate(self.layers):
            x = layer(x)                 # linear: xW^T + b
            x = self.activation(x)       # nonlinearity
        return x

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 200
    P: int = 51
    d: int = 5
    Sigma_scale: float = 0.08
    target_rho: float = 0.85

    gen_basis_type: str = "Bspline"
    gen_n_basis: int = 12
    gen_bspline_degree: int = 3
    map_mode: str = "nonlinear"   # "nonlinear" or "linear"
    map_bias: bool = True         # include intercept in linear map

    map_hidden: Tuple[int, ...] = (64, 64)
    map_activation: str = "tanh"
    map_weight_sd: float = 0.8

    meas_noise_sd: float = 0.5

def simulate_functional_ts(cfg: SimCfg) -> Dict[str, object]: # Functional Time Series 
    set_seed(cfg.seed)
    u = np.linspace(0.0, 1.0, cfg.P).astype(float)
    tpts = torch.tensor(u, dtype=torch.float32)

    Sigma = (cfg.Sigma_scale ** 2) * np.eye(cfg.d)
    Z_np, A_true = generate_latent_var1(T=cfg.T, d=cfg.d, Sigma=Sigma, seed=cfg.seed, target_rho=cfg.target_rho)
    Z = torch.tensor(Z_np, dtype=torch.float32)

    gen_builder = BasisFCBuilder(n_basis=cfg.gen_n_basis, basis_type=cfg.gen_basis_type, bspline_degree=cfg.gen_bspline_degree)
    Bgen = gen_builder.build(tpts)  # [P, M]

    torch.manual_seed(cfg.seed)

    if cfg.map_mode.lower() == "linear":
        mapper = LinearMap(d=cfg.d, hidden=[20], M=cfg.gen_n_basis, bias=True)

        for m in mapper.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=cfg.map_weight_sd)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    elif cfg.map_mode.lower() == "nonlinear":
        mapper = MLPMap(d=cfg.d, M=cfg.gen_n_basis, hidden=cfg.map_hidden, activation=cfg.map_activation)
        for m in mapper.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=cfg.map_weight_sd)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    else:
        raise ValueError("cfg.map_mode must be 'linear' or 'nonlinear'")

    mapper.eval()
    with torch.no_grad():
        coef = mapper(Z)                 # [T, M]
        X_clean = coef @ Bgen.T          # [T, P]
        X_noisy = X_clean + cfg.meas_noise_sd * torch.randn_like(X_clean)

    return {"u": u, "tpts": tpts, "X_clean": X_clean, "X_noisy": X_noisy, "A_true": A_true}


# ============================================================
# 3) FPCA 
# ============================================================

@torch.no_grad()
def uncentered_weighted_fpca_fit(X_train: torch.Tensor, w: torch.Tensor, K: int):
    """
    Uncentered FPCA/PCA:
      - NO mu, NO X - mu
      - Do SVD on weighted raw curves: Xw = X * sqrt(w)
    Returns:
      phi: [K, P] orthonormal under w
      scores: [T, K]
    """
    T, P = X_train.shape
    sw = torch.sqrt(w).view(1, -1)      # [1,P]
    Xw = X_train * sw                   # [T,P]

    U, S, Vh = torch.linalg.svd(Xw, full_matrices=False)
    V = Vh.transpose(0, 1)              # [P,r]

    phi = (V[:, :K] / sw.flatten()[:, None]).T  # [K,P]

    # normalize under w
    for k in range(K):
        nrm = torch.sqrt(torch.sum(phi[k] * phi[k] * w))
        phi[k] = phi[k] / nrm.clamp_min(1e-12)

    scores = (X_train * w.view(1, -1)) @ phi.T  # [T,K]
    return phi, scores

@torch.no_grad()
def fpca_reconstruct_uncentered_from_trainfit(X_train: torch.Tensor, X_eval: torch.Tensor, u: np.ndarray, K: int):
    w = trapezoid_weights(u)
    phi, scores_train = uncentered_weighted_fpca_fit(X_train, w, K)
    scores_eval = (X_eval * w.view(1, -1)) @ phi.T
    Xhat_eval = scores_eval @ phi
    return Xhat_eval, phi, scores_train


# ============================================================
# 4) VAR(1) helper (forecasts)  
# ============================================================

def fit_var1(H: np.ndarray, ridge: float = 1e-8):
    T, K = H.shape
    if T < 2:
        raise ValueError("Need at least 2 time points for VAR(1).")
    X = H[:-1, :]
    Y = H[1:, :]
    X_aug = np.hstack([np.ones((T - 1, 1)), X])
    XtX = X_aug.T @ X_aug
    B = np.linalg.solve(XtX + ridge * np.eye(XtX.shape[0]), X_aug.T @ Y)
    b = B[0, :]
    A = B[1:, :].T
    return b, A

def forecast_var1(h_last: np.ndarray, b: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    K = h_last.shape[0]
    out = np.zeros((steps, K), dtype=float)
    h = h_last.copy()
    for i in range(steps):
        h = b + (A @ h)
        out[i, :] = h
    return out


# ============================================================
# 5) FPCA+VAR forecast 
# ============================================================

@torch.no_grad()
def fpca_var_forecast_uncentered(X_train: torch.Tensor, u: np.ndarray, K: int, steps: int, var_ridge: float = 1e-8):
    w = trapezoid_weights(u)
    phi, scores = uncentered_weighted_fpca_fit(X_train, w, K)
    H = scores.detach().cpu().numpy()
    b, A = fit_var1(H, ridge=var_ridge)
    Hf = forecast_var1(H[-1], b, A, steps=steps)
    Hf_t = torch.tensor(Hf, dtype=torch.float32)
    X_fore = Hf_t @ phi
    return X_fore


# ============================================================
# 6) FAE 
# ============================================================

class FAEVanilla(nn.Module):
    def __init__(self, n_basis_project: int, n_rep: int, n_basis_revert: int, init_weight_sd: Optional[float] = None):
        super().__init__()
        self.fc1 = nn.Linear(n_basis_project, 100, bias=False)
        self.fc2 = nn.Linear(100, n_rep, bias=False)
        self.fc3 = nn.Linear(n_rep, 100, bias=False)
        self.fc4 = nn.Linear(100, n_basis_revert, bias=False)
        self.activation = nn.ReLU()

        if init_weight_sd is not None:
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.normal_(m.weight, mean=0.0, std=init_weight_sd)

    def project(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc: torch.Tensor) -> torch.Tensor:
        t = tpts.flatten()
        dt = t[1:] - t[:-1]
        zero = torch.zeros(1, device=x.device, dtype=x.dtype)
        W = 0.5 * torch.cat([zero, dt]) + 0.5 * torch.cat([dt, zero])  # [P]

        n_time = x.shape[1]
        if basis_fc.shape[0] == n_time:
            B = basis_fc
        elif basis_fc.shape[1] == n_time:
            B = basis_fc.T
        else:
            raise RuntimeError(f"basis_fc shape {tuple(basis_fc.shape)} not compatible with n_time={n_time}")

        return (x * W) @ B  # [batch, n_basis_project]

    def revert(self, coef: torch.Tensor, basis_fc: torch.Tensor) -> torch.Tensor:
        n_basis = coef.shape[1]
        if basis_fc.shape[1] == n_basis:
            return coef @ basis_fc.T
        elif basis_fc.shape[0] == n_basis:
            return coef @ basis_fc
        else:
            raise RuntimeError(f"basis_fc shape {tuple(basis_fc.shape)} not compatible with n_basis={n_basis}")

    def decode_from_rep(self, rep: torch.Tensor, basis_fc_revert: torch.Tensor) -> torch.Tensor:
        t2 = self.activation(self.fc3(rep))
        coef = self.fc4(t2)
        x_hat = self.revert(coef, basis_fc_revert)
        return x_hat

    def forward(self, x: torch.Tensor, tpts: torch.Tensor, basis_fc_project: torch.Tensor, basis_fc_revert: torch.Tensor):
        feature = self.project(x, tpts, basis_fc_project)
        t1 = self.activation(self.fc1(feature))
        rep = self.fc2(t1)
        t2 = self.activation(self.fc3(rep))
        coef = self.fc4(t2)
        x_hat = self.revert(coef, basis_fc_revert)
        return x_hat, rep, coef

def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    delta = coef[:, 2:] - 2 * coef[:, 1:-1] + coef[:, :-2]
    return torch.mean(torch.sum(delta ** 2, dim=1))

@dataclass
class FaeCfg:
    seed: int = 743
    device: str = "cpu"

    n_basis_project: int = 30
    n_basis_revert: int = 30
    basis_type_project: str = "Bspline"
    basis_type_revert: str = "Bspline"
    bspline_degree: int = 3

    n_rep: int = 5
    init_weight_sd: float = 0.5

    epochs: int = 2000
    batch_size: int = 16
    lr: float = 3e-4
    weight_decay: float = 1e-4
    split_rate: float = 0.85
    log_every: int = 200

    pen: str = "diff"
    lamb: float = 0.001

    var_ridge: float = 1e-8

def train_fae_on_noisy_train(Xn_train: torch.Tensor, tpts: torch.Tensor, cfg: FaeCfg):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    proj_builder = BasisFCBuilder(n_basis=cfg.n_basis_project, basis_type=cfg.basis_type_project, bspline_degree=cfg.bspline_degree)
    rev_builder  = BasisFCBuilder(n_basis=cfg.n_basis_revert,  basis_type=cfg.basis_type_revert,  bspline_degree=cfg.bspline_degree)
    Bp = proj_builder.build(tpts.to(device)).to(device)
    Br = rev_builder.build(tpts.to(device)).to(device)
    tpts_d = tpts.to(device).float()

    loss_fn = nn.MSELoss()

    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(idx_all, train_size=cfg.split_rate, random_state=cfg.seed, shuffle=True)
    TrainData = Xn_train[idx_tr].float()
    ValData   = Xn_train[idx_va].float()

    loader = DataLoader(TrainData, batch_size=cfg.batch_size, shuffle=True)

    model = FAEVanilla(cfg.n_basis_project, cfg.n_rep, cfg.n_basis_revert, init_weight_sd=cfg.init_weight_sd).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    for ep in range(1, cfg.epochs + 1):
        model.train()
        for xb in loader:
            xb = xb.to(device).float()
            opt.zero_grad()
            xhat, rep, coef = model(xb, tpts_d, Bp, Br)
            loss = loss_fn(xhat, xb)
            if cfg.pen == "diff" and cfg.lamb > 0:
                loss = loss + cfg.lamb * diff_penalty(coef)
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            model.eval()
            with torch.no_grad():
                tr_hat, _, _ = model(TrainData.to(device), tpts_d, Bp, Br)
                va_hat, _, _ = model(ValData.to(device),   tpts_d, Bp, Br)
                tr = float(loss_fn(tr_hat, TrainData.to(device)).detach().cpu())
                va = float(loss_fn(va_hat, ValData.to(device)).detach().cpu())
            print(f"[FAE] ep {ep:4d} | train_mse(noisy)={tr:.6e} | val_mse(noisy)={va:.6e}")

    return model, tpts_d.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()

@torch.no_grad()
def fae_reconstruct(model: FAEVanilla, X: torch.Tensor, tpts: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    device = next(model.parameters()).device
    X = X.to(device).float()
    tpts = tpts.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)
    Xhat, H, _ = model(X, tpts, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu()

@torch.no_grad()
def fae_var_forecast(model: FAEVanilla, Xn_train: torch.Tensor, steps: int, tpts: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor, var_ridge: float):
    _, H_train = fae_reconstruct(model, Xn_train, tpts, Bp, Br)
    H_np = H_train.numpy()
    b, A = fit_var1(H_np, ridge=var_ridge)
    Hf = forecast_var1(H_np[-1], b, A, steps=steps)

    device = next(model.parameters()).device
    Hf_t = torch.tensor(Hf, dtype=torch.float32, device=device)
    Br_d = Br.to(device)
    X_fore = model.decode_from_rep(Hf_t, Br_d).detach().cpu()
    return X_fore


# ============================================================
# 7) Run 
# ============================================================

@dataclass
class RunCfg:
    horizon: int = 5
    fpca_K: int = 5
    fpca_var_ridge: float = 1e-8

def run_two_reports(sim_cfg: SimCfg, run_cfg: RunCfg, fae_cfg: FaeCfg) -> None:
    sim = simulate_functional_ts(sim_cfg)
    u = sim["u"]
    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    print("Shapes:")
    print("  train:", tuple(Xn_train.shape), " test:", tuple(Xn_test.shape), " horizon:", run_cfg.horizon)

    # FPCA recon
    X_fpca_recon_train, phi_fpca, scores_train = fpca_reconstruct_uncentered_from_trainfit(
        X_train=Xn_train, X_eval=Xn_train, u=u, K=run_cfg.fpca_K
    )

    # FAE recon
    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_noisy_train(Xn_train, tpts, fae_cfg)
    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, tpts_fae, Bp_fae, Br_fae)

    recon_rows = {
        "FPCA": {
            "relMSE_vs_clean": rel_mse(X_fpca_recon_train, Xc_train),
            "relMSE_vs_noisy": rel_mse(X_fpca_recon_train, Xn_train),
        },
        "FAE": {
            "relMSE_vs_clean": rel_mse(X_fae_recon_train, Xc_train),
            "relMSE_vs_noisy": rel_mse(X_fae_recon_train, Xn_train),
        },
    }
    print_report_table("REPORT 1 — Reconstruction on TRAIN", recon_rows)

    # Forecast
    X_fpca_fore = fpca_var_forecast_uncentered(
        X_train=Xn_train, u=u, K=run_cfg.fpca_K, steps=run_cfg.horizon, var_ridge=run_cfg.fpca_var_ridge
    )

    X_fae_fore = fae_var_forecast(
        model=model_fae,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        tpts=tpts_fae,
        Bp= Bp_fae,
        Br= Br_fae,
        var_ridge=fae_cfg.var_ridge,
    )

    fore_rows = {
        "FPCA+VAR": {
            "relMSE_vs_clean": rel_mse(X_fpca_fore, Xc_test),
            "relMSE_vs_noisy": rel_mse(X_fpca_fore, Xn_test),
        },
        "FAE+VAR": {
            "relMSE_vs_clean": rel_mse(X_fae_fore, Xc_test),
            "relMSE_vs_noisy": rel_mse(X_fae_fore, Xn_test),
        },
    }
    print_report_table(f"REPORT 2 — Forecast on TEST horizon (H={run_cfg.horizon})", fore_rows)


# ============================================================
# 8) Batch runner to Excel
# ============================================================

@torch.no_grad()
def _mse(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.mean((a - b) ** 2).detach().cpu())

def _build_fixed_decoder(sim_cfg: SimCfg) -> Dict[str, object]:
    """
    Build a fixed simulator 'decoder' ONCE (same mapper weights, same basis, same VAR(1) matrix A_true),
    so that each repetition uses a new dataset generated with new VAR noise only.
    """
    set_seed(sim_cfg.seed)

    u = np.linspace(0.0, 1.0, sim_cfg.P).astype(float)
    tpts = torch.tensor(u, dtype=torch.float32)

    Sigma = (sim_cfg.Sigma_scale ** 2) * np.eye(sim_cfg.d)

    # basis fixed
    gen_builder = BasisFCBuilder(
        n_basis=sim_cfg.gen_n_basis,
        basis_type=sim_cfg.gen_basis_type,
        bspline_degree=sim_cfg.gen_bspline_degree,
    )
    Bgen = gen_builder.build(tpts)  # [P, M]

    # A_true fixed 
    _, A_true = generate_latent_var1(
        T=sim_cfg.T, d=sim_cfg.d, Sigma=Sigma, seed=sim_cfg.seed, target_rho=sim_cfg.target_rho
    )

    # mapper fixed 
    torch.manual_seed(sim_cfg.seed)

    if sim_cfg.map_mode.lower() == "linear":
        mapper = LinearMap(d=sim_cfg.d, hidden=[20], M=sim_cfg.gen_n_basis, bias=True)
        for m in mapper.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    elif sim_cfg.map_mode.lower() == "nonlinear":
        mapper = MLPMap(d=sim_cfg.d, M=sim_cfg.gen_n_basis, hidden=sim_cfg.map_hidden, activation=sim_cfg.map_activation)
        for m in mapper.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    else:
        raise ValueError("cfg.map_mode must be 'linear' or 'nonlinear'")

    mapper.eval()
    for p in mapper.parameters():
        p.requires_grad_(False)

    return {
        "u": u,
        "tpts": tpts,
        "Sigma": Sigma,
        "A_true": A_true,
        "Bgen": Bgen,
        "mapper": mapper,
    }

def _simulate_new_dataset_same_decoder(fixed: Dict[str, object], sim_cfg: SimCfg, data_seed: int):
    """
    New dataset each time:
      - same A_true, same mapper, same basis
      - new VAR(1) innovations via data_seed

    Inputs:
        - fixed: stuff we decided to keep the same every rep (A matrix, mapper NN, basis, Sigma)
        - sim_cfg: simulation settings (T, d, noise level)
        - data_seed: changes each rep to make a new dataset
    """
    rng = np.random.default_rng(data_seed) # Creates a NumPy random generator seeded with data_seed.
    A = fixed["A_true"] # VAR(1) transition matrix (fixed across reps)
    Sigma = fixed["Sigma"] # Sigma: the covariance of the innovations (fixed)

    T, d = sim_cfg.T, sim_cfg.d # time length T and latent dimension d
    Z = np.zeros((T, d), dtype=float)  # an empty array for the latent states: shape (T, d)
    # Initialize the first latent vector Z[0] by sampling from a multivariate normal: mean = 0 vector, covariance = Sigma: 
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma) 
    for t in range(1, T):
        eps = rng.multivariate_normal(np.zeros(d), Sigma) # New noise vector eps ~ N(0, Sigma)
        Z[t] = A @ Z[t - 1] + eps # update latent state using VAR(1)

    Z_t = torch.tensor(Z, dtype=torch.float32)

    # Decode latent Z into functional curves:
    mapper: nn.Module = fixed["mapper"] # neural network that maps Z_t → coefficients
    Bgen: torch.Tensor = fixed["Bgen"] # basis matrix evaluated on the time grid

    with torch.no_grad(): # don’t track gradients (simulating data, not training)
        coef = mapper(Z_t)            # [T, M]
        X_clean = coef @ Bgen.T       # [T, P] Coeff * Basis 
        X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)

    return X_clean, X_noisy

def _run_one_rep_and_collect(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    rep: int,
    data_seed_base: int,
) -> Dict[str, float]:
    """
    One repetition:
      - new dataset (same decoder)
      - FPCA recon + FAE recon on TRAIN
      - FPCA+VAR forecast + FAE+VAR forecast on TEST
      - final FAE train/val MSE (noisy target)
    """
    data_seed = data_seed_base + rep

    X_clean, X_noisy = _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=data_seed)
    u = fixed["u"]
    tpts = fixed["tpts"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    # REPORT 1: FPCA recon
    X_fpca_recon_train, _, _ = fpca_reconstruct_uncentered_from_trainfit(
        X_train=Xn_train, X_eval=Xn_train, u=u, K=run_cfg.fpca_K
    )

    # REPORT 1: FAE recon (train a new FAE each rep; seed fixed inside train_fae_on_noisy_train)
    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_noisy_train(Xn_train, tpts, fae_cfg)
    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, tpts_fae, Bp_fae, Br_fae)

    # final train/val MSE (noisy target), computed using same split rule
    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(idx_all, train_size=fae_cfg.split_rate, random_state=fae_cfg.seed, shuffle=True)
    TrainData = Xn_train[idx_tr].float()
    ValData   = Xn_train[idx_va].float()

    device = torch.device(fae_cfg.device)
    model_fae.eval()
    with torch.no_grad():
        tr_hat, _, _ = model_fae(TrainData.to(device), tpts_fae.to(device), Bp_fae.to(device), Br_fae.to(device))
        va_hat, _, _ = model_fae(ValData.to(device),   tpts_fae.to(device), Bp_fae.to(device), Br_fae.to(device))

    fae_train_mse_final = _mse(tr_hat.detach().cpu(), TrainData)
    fae_val_mse_final   = _mse(va_hat.detach().cpu(), ValData)

    # reconstruction relMSE
    fpca_recon_rel_clean = rel_mse(X_fpca_recon_train, Xc_train)
    fpca_recon_rel_noisy = rel_mse(X_fpca_recon_train, Xn_train)
    fae_recon_rel_clean  = rel_mse(X_fae_recon_train,  Xc_train)
    fae_recon_rel_noisy  = rel_mse(X_fae_recon_train,  Xn_train)

    # REPORT 2: forecasts
    X_fpca_fore = fpca_var_forecast_uncentered(
        X_train=Xn_train, u=u, K=run_cfg.fpca_K, steps=run_cfg.horizon, var_ridge=run_cfg.fpca_var_ridge
    )
    X_fae_fore = fae_var_forecast(
        model=model_fae,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        tpts=tpts_fae,
        Bp=Bp_fae,
        Br=Br_fae,
        var_ridge=fae_cfg.var_ridge,
    )

    fpca_fore_rel_clean = rel_mse(X_fpca_fore, Xc_test)
    fpca_fore_rel_noisy = rel_mse(X_fpca_fore, Xn_test)
    fae_fore_rel_clean  = rel_mse(X_fae_fore,  Xc_test)
    fae_fore_rel_noisy  = rel_mse(X_fae_fore,  Xn_test)

    return {
        "rep": rep,
        "data_seed": data_seed,

        # REPORT 1 — Reconstruction on TRAIN
        "fpca_recon_relMSE_vs_clean": fpca_recon_rel_clean,
        "fpca_recon_relMSE_vs_noisy": fpca_recon_rel_noisy,
        "fae_recon_relMSE_vs_clean":  fae_recon_rel_clean,
        "fae_recon_relMSE_vs_noisy":  fae_recon_rel_noisy,

        # Final FAE train/val MSE (noisy target)
        "fae_train_mse_final_noisy": fae_train_mse_final,
        "fae_val_mse_final_noisy":   fae_val_mse_final,

        # REPORT 2 — Forecast on TEST horizon
        "fpca_fore_relMSE_vs_clean": fpca_fore_rel_clean,
        "fpca_fore_relMSE_vs_noisy": fpca_fore_rel_noisy,
        "fae_fore_relMSE_vs_clean":  fae_fore_rel_clean,
        "fae_fore_relMSE_vs_noisy":  fae_fore_rel_noisy,
    }

def run_sim_fpca_fae_many_to_excel(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 100,
    out_xlsx_path: str = "results.xlsx",
    data_seed_base: int = 20000,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Runs your pipeline n_reps times and saves an Excel file.
      - Each repetition uses a NEW dataset (new VAR noise seed).
      - The simulation decoder (A_true + mapper + basis) is fixed across repetitions.
      - FAE is trained fresh each repetition, but uses the same cfg.seed (so same init/split each time).
    """
    fixed = _build_fixed_decoder(sim_cfg)

    rows: List[Dict[str, float]] = []
    for rep in range(1, n_reps + 1):
        if not verbose:
            # silence training prints by temporarily setting log_every=0
            old_log = fae_cfg.log_every
            fae_cfg.log_every = 0
        try:
            row = _run_one_rep_and_collect(
                fixed=fixed,
                sim_cfg=sim_cfg,
                run_cfg=run_cfg,
                fae_cfg=fae_cfg,
                rep=rep,
                data_seed_base=data_seed_base,
            )
        finally:
            if not verbose:
                fae_cfg.log_every = old_log

        # add config tags for convenience
        row.update({
            "map_mode": sim_cfg.map_mode,
            "gen_basis_type": sim_cfg.gen_basis_type,
            "gen_n_basis": sim_cfg.gen_n_basis,
            "T": sim_cfg.T,
            "P": sim_cfg.P,
            "d": sim_cfg.d,
            "meas_noise_sd": sim_cfg.meas_noise_sd,
            "fpca_K": run_cfg.fpca_K,
            "horizon": run_cfg.horizon,
            "fae_n_rep": fae_cfg.n_rep,
            "fae_n_basis_project": fae_cfg.n_basis_project,
            "fae_n_basis_revert": fae_cfg.n_basis_revert,
        })
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_excel(out_xlsx_path, index=False)
    return df


In [2]:

# ============================================================
# 9) Example: 100 runs 
# ============================================================

if __name__ == "__main__":
    sim_cfg = SimCfg(
        seed=123,
        T=500,
        P=100,
        d=5,
        Sigma_scale=0.08,
        target_rho=0.85,
        gen_basis_type="Bspline",
        gen_n_basis=12,
        gen_bspline_degree=3,
        map_mode="nonlinear",  # or "linear"
        map_hidden=(64, 64),
        map_activation="relu",
        map_weight_sd=0.8,
        meas_noise_sd=0.5,
    )

    run_cfg = RunCfg(
        horizon=5,
        fpca_K=5,
        fpca_var_ridge=1e-8,
    )

    fae_cfg = FaeCfg(
        seed=743,
        device="cpu",
        n_basis_project=30,
        n_basis_revert=30,
        basis_type_project="Bspline",
        basis_type_revert="Bspline",
        bspline_degree=3,
        n_rep=5,
        init_weight_sd=0.5,
        epochs=2000,
        batch_size=16,
        lr=3e-4,
        weight_decay=1e-4,
        split_rate=0.85,
        log_every=200,   
        pen="diff",
        lamb=0.001,
        var_ridge=1e-8,
    )

    df = run_sim_fpca_fae_many_to_excel(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        out_xlsx_path="sim_results_10runs_new.xlsx",
        data_seed_base=30000,
        verbose=True,   #  FAE epoch prints
    )

    df.head()

[FAE] ep  200 | train_mse(noisy)=4.170232e-01 | val_mse(noisy)=4.438848e-01
[FAE] ep  400 | train_mse(noisy)=3.723221e-01 | val_mse(noisy)=3.877058e-01
[FAE] ep  600 | train_mse(noisy)=3.469739e-01 | val_mse(noisy)=3.724085e-01
[FAE] ep  800 | train_mse(noisy)=3.356712e-01 | val_mse(noisy)=3.632661e-01
[FAE] ep 1000 | train_mse(noisy)=3.286296e-01 | val_mse(noisy)=3.627887e-01
[FAE] ep 1200 | train_mse(noisy)=3.229918e-01 | val_mse(noisy)=3.604638e-01
[FAE] ep 1400 | train_mse(noisy)=3.184374e-01 | val_mse(noisy)=3.569276e-01
[FAE] ep 1600 | train_mse(noisy)=3.146721e-01 | val_mse(noisy)=3.559562e-01
[FAE] ep 1800 | train_mse(noisy)=3.122607e-01 | val_mse(noisy)=3.543962e-01
[FAE] ep 2000 | train_mse(noisy)=3.092702e-01 | val_mse(noisy)=3.525467e-01
[FAE] ep  200 | train_mse(noisy)=4.201055e-01 | val_mse(noisy)=4.370127e-01
[FAE] ep  400 | train_mse(noisy)=3.799853e-01 | val_mse(noisy)=3.897390e-01
[FAE] ep  600 | train_mse(noisy)=3.516836e-01 | val_mse(noisy)=3.633383e-01
[FAE] ep  80

In [10]:
# ============================================================
# Functional time series simulation:
# FPCA + linear VAR baseline vs FAE + nonlinear forecast head
# ============================================================

import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import SplineTransformer


# ============================================================
# 0) Repro
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ============================================================
# 1) Configs
# ============================================================

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 500
    P: int = 100
    d: int = 5

    # latent innovation covariance scale
    Sigma_scale: float = 0.08
    target_rho: float = 0.80

    # generator basis
    gen_basis_type: str = "bspline"   # "bspline" or "fourier"
    gen_n_basis: int = 12
    gen_bspline_degree: int = 3

    # nonlinear decoder from latent state -> basis coefficients
    map_hidden: Tuple[int, int] = (64, 64)
    map_activation: str = "relu"
    map_weight_sd: float = 0.8

    # observation noise
    meas_noise_sd: float = 0.25

    # NEW: nonlinear latent dynamics strength
    latent_nl_scale: float = 0.25
    latent_hetero_scale: float = 0.15


@dataclass
class RunCfg:
    horizon: int = 10
    fpca_K: int = 5
    fpca_var_ridge: float = 1e-6


@dataclass
class FaeCfg:
    seed: int = 743
    device: str = "cpu"

    n_basis_project: int = 30
    n_basis_revert: int = 30
    basis_type_project: str = "bspline"
    basis_type_revert: str = "bspline"
    bspline_degree: int = 3

    n_rep: int = 8   # latent representation size
    init_weight_sd: float = 0.3

    ae_epochs: int = 1200
    dyn_epochs: int = 800

    batch_size: int = 32
    lr_ae: float = 3e-4
    lr_dyn: float = 3e-4
    weight_decay: float = 1e-4

    split_rate: float = 0.85
    log_every: int = 100

    # optional coefficient smoothness penalty
    lamb: float = 1e-4

    # multi-step rollout training
    dyn_teacher_forcing_noise: float = 0.01


# ============================================================
# 2) Small utilities
# ============================================================

def _mse(a, b) -> float:
    a = torch.as_tensor(a, dtype=torch.float32)
    b = torch.as_tensor(b, dtype=torch.float32)
    return float(torch.mean((a - b) ** 2).item())

def rel_mse(pred, truth) -> float:
    pred = torch.as_tensor(pred, dtype=torch.float32)
    truth = torch.as_tensor(truth, dtype=torch.float32)
    num = torch.mean((pred - truth) ** 2)
    den = torch.mean(truth ** 2) + 1e-12
    return float((num / den).item())

def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    return X[:-horizon], X[-horizon:]

def activation_from_name(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


# ============================================================
# 3) Basis builders
# ============================================================

def make_grid(P: int) -> np.ndarray:
    return np.linspace(0.0, 1.0, P)

def make_bspline_basis(u: np.ndarray, n_basis: int, degree: int) -> np.ndarray:
    # sklearn SplineTransformer returns basis matrix [P, n_features]
    st = SplineTransformer(
        n_knots=max(n_basis - degree + 1, degree + 1),
        degree=degree,
        include_bias=True
    )
    B = st.fit_transform(u.reshape(-1, 1))
    if B.shape[1] > n_basis:
        B = B[:, :n_basis]
    elif B.shape[1] < n_basis:
        pad = np.zeros((B.shape[0], n_basis - B.shape[1]))
        B = np.hstack([B, pad])
    return B.astype(np.float32)

def make_fourier_basis(u: np.ndarray, n_basis: int) -> np.ndarray:
    cols = [np.ones_like(u)]
    k = 1
    while len(cols) < n_basis:
        cols.append(np.sin(2 * np.pi * k * u))
        if len(cols) < n_basis:
            cols.append(np.cos(2 * np.pi * k * u))
        k += 1
    B = np.column_stack(cols[:n_basis]).astype(np.float32)
    return B

def make_basis(u: np.ndarray, basis_type: str, n_basis: int, degree: int = 3) -> np.ndarray:
    basis_type = basis_type.lower()
    if basis_type == "bspline":
        return make_bspline_basis(u, n_basis, degree)
    if basis_type == "fourier":
        return make_fourier_basis(u, n_basis)
    raise ValueError(f"Unknown basis_type: {basis_type}")


# ============================================================
# 4) Generator pieces
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: Tuple[int, ...], out_dim: int, act: str = "relu"):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(activation_from_name(act))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def stabilize_A(A: np.ndarray, target_rho: float = 0.8) -> np.ndarray:
    eigvals = np.linalg.eigvals(A)
    rho = float(np.max(np.abs(eigvals)))
    if rho < 1e-12:
        return A
    return A * (target_rho / rho)


def generate_latent_nonlinear_ar1(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    target_rho: float = 0.8,
    nl_scale: float = 0.25,
    hetero_scale: float = 0.15,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Mildly nonlinear latent dynamics.
    Still close to AR(1), but no longer ideal for linear FPCA forecasting.

    z_t = A z_{t-1}
          + nonlinear_drift(z_{t-1})
          + state-dependent noise
    """
    rng = np.random.default_rng(seed)

    A = rng.normal(size=(d, d)) * 0.20
    A = stabilize_A(A, target_rho=target_rho)

    Z = np.zeros((T, d), dtype=np.float32)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma).astype(np.float32)

    for t in range(1, T):
        zprev = Z[t - 1].copy()
        Az = A @ zprev

        # Local nonlinear drift on first coordinates
        drift = np.zeros(d, dtype=np.float32)
        if d >= 1:
            drift[0] += nl_scale * (zprev[0] ** 2 - 0.75)
        if d >= 2:
            drift[1] += 0.8 * nl_scale * np.sin(1.5 * zprev[1])
        if d >= 3:
            drift[2] += 0.6 * nl_scale * np.tanh(zprev[0] * zprev[2])
        if d >= 4:
            drift[3] += 0.5 * nl_scale * (zprev[1] * zprev[3])

        scale = 1.0 + hetero_scale * abs(float(zprev[0]))
        eps = rng.multivariate_normal(np.zeros(d), (scale ** 2) * Sigma).astype(np.float32)

        Z[t] = Az + drift + eps

    return Z, A


def build_fixed_decoder(sim_cfg: SimCfg) -> Dict[str, object]:
    set_seed(sim_cfg.seed)

    u = make_grid(sim_cfg.P)
    Bgen_np = make_basis(
        u=u,
        basis_type=sim_cfg.gen_basis_type,
        n_basis=sim_cfg.gen_n_basis,
        degree=sim_cfg.gen_bspline_degree
    )

    mapper = MLP(
        in_dim=sim_cfg.d,
        hidden=sim_cfg.map_hidden,
        out_dim=sim_cfg.gen_n_basis,
        act=sim_cfg.map_activation,
    )

    for m in mapper.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)

    return {
        "u": torch.tensor(u, dtype=torch.float32),
        "tpts": torch.tensor(u, dtype=torch.float32),
        "Bgen": torch.tensor(Bgen_np, dtype=torch.float32),
        "mapper": mapper.eval(),
    }


def simulate_functional_ts(sim_cfg: SimCfg) -> Dict[str, torch.Tensor]:
    fixed = build_fixed_decoder(sim_cfg)
    return _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=sim_cfg.seed + 1000)


def _simulate_new_dataset_same_decoder(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    data_seed: int,
) -> Dict[str, torch.Tensor]:
    Sigma = (sim_cfg.Sigma_scale ** 2) * np.eye(sim_cfg.d, dtype=np.float32)

    Z, A_true = generate_latent_nonlinear_ar1(
        T=sim_cfg.T,
        d=sim_cfg.d,
        Sigma=Sigma,
        seed=data_seed,
        target_rho=sim_cfg.target_rho,
        nl_scale=sim_cfg.latent_nl_scale,
        hetero_scale=sim_cfg.latent_hetero_scale,
    )

    Z_t = torch.tensor(Z, dtype=torch.float32)

    mapper: nn.Module = fixed["mapper"]
    Bgen: torch.Tensor = fixed["Bgen"]

    with torch.no_grad():
        coef = mapper(Z_t)                   # [T, M]
        X_clean = coef @ Bgen.T              # [T, P]
        X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)

    return {
        "u": fixed["u"],
        "tpts": fixed["tpts"],
        "Z": Z_t,
        "A_true": torch.tensor(A_true, dtype=torch.float32),
        "X_clean": X_clean,
        "X_noisy": X_noisy,
    }


# ============================================================
# 5) FPCA baseline
# ============================================================

def fit_var1(Y: np.ndarray, ridge: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    """
    Y_t = b + A Y_{t-1}
    """
    X = Y[:-1]
    Z = Y[1:]
    X1 = np.hstack([np.ones((X.shape[0], 1)), X])  # [n, 1+p]

    XtX = X1.T @ X1
    reg = ridge * np.eye(X1.shape[1], dtype=np.float64)
    B = np.linalg.solve(XtX + reg, X1.T @ Z)       # [1+p, p]

    b = B[0]
    A = B[1:].T
    return b.astype(np.float32), A.astype(np.float32)

def forecast_var1(y_last: np.ndarray, b: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    out = []
    cur = y_last.astype(np.float32).copy()
    for _ in range(steps):
        cur = b + A @ cur
        out.append(cur.copy())
    return np.stack(out, axis=0)

def fpca_reconstruct_uncentered_from_trainfit(
    X_train: torch.Tensor,
    X_eval: torch.Tensor,
    K: int,
) -> Tuple[torch.Tensor, PCA, np.ndarray]:
    pca = PCA(n_components=K, svd_solver="full")
    scores_train = pca.fit_transform(X_train.numpy())
    scores_eval = pca.transform(X_eval.numpy())
    Xhat = pca.inverse_transform(scores_eval)
    return torch.tensor(Xhat, dtype=torch.float32), pca, scores_train

def fpca_var_forecast_uncentered(
    X_train: torch.Tensor,
    K: int,
    steps: int,
    var_ridge: float = 1e-6
) -> torch.Tensor:
    pca = PCA(n_components=K, svd_solver="full")
    scores = pca.fit_transform(X_train.numpy())
    b, A = fit_var1(scores, ridge=var_ridge)
    Sf = forecast_var1(scores[-1], b, A, steps)
    Xf = pca.inverse_transform(Sf)
    return torch.tensor(Xf, dtype=torch.float32)


# ============================================================
# 6) FAE model
# ============================================================

class FAEVanilla(nn.Module):
    """
    Project x(t) onto basis Bp -> coefficients -> encoder -> latent rep
    latent rep -> decoder -> coefficients on Br -> reconstruct curve
    """
    def __init__(
        self,
        P: int,
        n_basis_project: int,
        n_basis_revert: int,
        n_rep: int,
        hidden: Tuple[int, int] = (64, 64),
    ):
        super().__init__()

        self.enc = nn.Sequential(
            nn.Linear(n_basis_project, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

        self.dec = nn.Sequential(
            nn.Linear(n_rep, hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], n_basis_revert),
        )

        self.P = P
        self.n_basis_project = n_basis_project
        self.n_basis_revert = n_basis_revert
        self.n_rep = n_rep

    def encode_from_curve(self, X: torch.Tensor, Bp: torch.Tensor) -> torch.Tensor:
        # least-squares projection onto basis
        # coef = argmin ||X - coef Bp^T||^2
        G = Bp.T @ Bp + 1e-6 * torch.eye(Bp.shape[1], device=Bp.device)
        coef = torch.linalg.solve(G, Bp.T @ X.T).T
        H = self.enc(coef)
        return H

    def decode_from_rep(self, H: torch.Tensor, Br: torch.Tensor) -> torch.Tensor:
        coef = self.dec(H)
        Xhat = coef @ Br.T
        return Xhat

    def forward(self, X: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor):
        H = self.encode_from_curve(X, Bp)
        coef_r = self.dec(H)
        Xhat = coef_r @ Br.T
        return Xhat, H, coef_r


class LatentDynamicsMLP(nn.Module):
    """
    One-step latent transition: h_{t+1} = g(h_t)
    """
    def __init__(self, n_rep: int, hidden: Tuple[int, int] = (64, 64)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_rep, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

    def forward(self, h):
        return self.net(h)


def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    if coef.shape[1] <= 1:
        return torch.tensor(0.0, device=coef.device)
    d1 = coef[:, 1:] - coef[:, :-1]
    return torch.mean(d1 ** 2)


def init_small_weights(model: nn.Module, sd: float) -> None:
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)


def train_fae_on_noisy_train(
    Xn_train: torch.Tensor,
    tpts: torch.Tensor,
    cfg: FaeCfg,
) -> Tuple[FAEVanilla, torch.Tensor, torch.Tensor, torch.Tensor]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    P = Xn_train.shape[1]
    u = tpts.numpy()

    Bp_np = make_basis(u, cfg.basis_type_project, cfg.n_basis_project, cfg.bspline_degree)
    Br_np = make_basis(u, cfg.basis_type_revert, cfg.n_basis_revert, cfg.bspline_degree)

    Bp = torch.tensor(Bp_np, dtype=torch.float32, device=device)
    Br = torch.tensor(Br_np, dtype=torch.float32, device=device)

    model = FAEVanilla(
        P=P,
        n_basis_project=cfg.n_basis_project,
        n_basis_revert=cfg.n_basis_revert,
        n_rep=cfg.n_rep,
        hidden=(64, 64),
    ).to(device)

    init_small_weights(model, cfg.init_weight_sd)

    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=cfg.split_rate,
        random_state=cfg.seed,
        shuffle=True
    )

    Xtr = Xn_train[idx_tr].to(device).float()
    Xva = Xn_train[idx_va].to(device).float()

    opt = optim.Adam(model.parameters(), lr=cfg.lr_ae, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.ae_epochs + 1):
        model.train()
        perm = torch.randperm(Xtr.shape[0], device=device)

        for i in range(0, Xtr.shape[0], cfg.batch_size):
            batch = Xtr[perm[i:i + cfg.batch_size]]
            opt.zero_grad()
            xhat, _, coef = model(batch, Bp, Br)
            loss = loss_fn(xhat, batch) + cfg.lamb * diff_penalty(coef)
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            model.eval()
            with torch.no_grad():
                tr_hat, _, _ = model(Xtr, Bp, Br)
                va_hat, _, _ = model(Xva, Bp, Br)
                tr = loss_fn(tr_hat, Xtr).item()
                va = loss_fn(va_hat, Xva).item()
            print(f"[AE ] ep {ep:4d} | train_mse={tr:.6e} | val_mse={va:.6e}")

    return model.eval(), tpts.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()


@torch.no_grad()
def fae_reconstruct(
    model: FAEVanilla,
    X: torch.Tensor,
    Bp: torch.Tensor,
    Br: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    device = next(model.parameters()).device
    X = X.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)
    Xhat, H, _ = model(X, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu()


def train_fae_dynamics_head(
    model: FAEVanilla,
    Xn_train: torch.Tensor,
    Bp: torch.Tensor,
    cfg: FaeCfg,
) -> LatentDynamicsMLP:
    """
    Train nonlinear one-step latent transition on encoded FAE states.
    """
    set_seed(cfg.seed + 17)
    device = next(model.parameters()).device
    Bp = Bp.to(device)

    model.eval()
    with torch.no_grad():
        H = model.encode_from_curve(Xn_train.to(device).float(), Bp)

    H_in = H[:-1]
    H_out = H[1:]

    dyn = LatentDynamicsMLP(n_rep=cfg.n_rep, hidden=(64, 64)).to(device)
    init_small_weights(dyn, cfg.init_weight_sd)

    opt = optim.Adam(dyn.parameters(), lr=cfg.lr_dyn, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.dyn_epochs + 1):
        dyn.train()
        perm = torch.randperm(H_in.shape[0], device=device)

        for i in range(0, H_in.shape[0], cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            hin = H_in[idx]
            hout = H_out[idx]

            if cfg.dyn_teacher_forcing_noise > 0:
                hin = hin + cfg.dyn_teacher_forcing_noise * torch.randn_like(hin)

            pred = dyn(hin)
            loss = loss_fn(pred, hout)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            dyn.eval()
            with torch.no_grad():
                pred = dyn(H_in)
                tr = loss_fn(pred, H_out).item()
            print(f"[DYN] ep {ep:4d} | one_step_latent_mse={tr:.6e}")

    return dyn.eval()


@torch.no_grad()
def fae_nonlinear_forecast(
    model: FAEVanilla,
    dyn: LatentDynamicsMLP,
    Xn_train: torch.Tensor,
    steps: int,
    Bp: torch.Tensor,
    Br: torch.Tensor,
) -> torch.Tensor:
    device = next(model.parameters()).device
    Bp = Bp.to(device)
    Br = Br.to(device)

    H = model.encode_from_curve(Xn_train.to(device).float(), Bp)
    h = H[-1:].clone()

    preds = []
    for _ in range(steps):
        h = dyn(h)
        xhat = model.decode_from_rep(h, Br)
        preds.append(xhat.squeeze(0).detach().cpu())

    return torch.stack(preds, dim=0)


# ============================================================
# 7) One run
# ============================================================

def run_one_rep(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    rep: int,
    data_seed_base: int,
    verbose: bool = False,
) -> Dict[str, float]:
    data_seed = data_seed_base + rep
    sim = _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=data_seed)

    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    # ---------- FPCA reconstruction ----------
    X_fpca_recon_train, _, _ = fpca_reconstruct_uncentered_from_trainfit(
        X_train=Xn_train,
        X_eval=Xn_train,
        K=run_cfg.fpca_K,
    )

    # ---------- FAE reconstruction ----------
    old_log = fae_cfg.log_every
    if not verbose:
        fae_cfg.log_every = 0

    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_noisy_train(Xn_train, tpts, fae_cfg)
    Bp_dev = Bp_fae.to(fae_cfg.device)
    Br_dev = Br_fae.to(fae_cfg.device)

    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, Bp_dev, Br_dev)

    # ---------- Forecast ----------
    X_fpca_fore = fpca_var_forecast_uncentered(
        X_train=Xn_train,
        K=run_cfg.fpca_K,
        steps=run_cfg.horizon,
        var_ridge=run_cfg.fpca_var_ridge,
    )

    dyn_head = train_fae_dynamics_head(model_fae, Xn_train, Bp_dev, fae_cfg)
    X_fae_fore = fae_nonlinear_forecast(
        model=model_fae,
        dyn=dyn_head,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        Bp=Bp_dev,
        Br=Br_dev,
    )

    fae_cfg.log_every = old_log

    # ---------- Metrics ----------
    # Reconstruction
    fpca_recon_rel_clean = rel_mse(X_fpca_recon_train, Xc_train)
    fpca_recon_rel_noisy = rel_mse(X_fpca_recon_train, Xn_train)
    fae_recon_rel_clean = rel_mse(X_fae_recon_train, Xc_train)
    fae_recon_rel_noisy = rel_mse(X_fae_recon_train, Xn_train)

    # Forecast
    fpca_fore_rel_clean = rel_mse(X_fpca_fore, Xc_test)
    fpca_fore_rel_noisy = rel_mse(X_fpca_fore, Xn_test)
    fae_fore_rel_clean = rel_mse(X_fae_fore, Xc_test)
    fae_fore_rel_noisy = rel_mse(X_fae_fore, Xn_test)

    # Final AE train/val MSE
    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=fae_cfg.split_rate,
        random_state=fae_cfg.seed,
        shuffle=True
    )
    Xtr = Xn_train[idx_tr].float()
    Xva = Xn_train[idx_va].float()

    model_fae.eval()
    with torch.no_grad():
        xtr_hat, _, _ = model_fae(Xtr.to(fae_cfg.device), Bp_dev, Br_dev)
        xva_hat, _, _ = model_fae(Xva.to(fae_cfg.device), Bp_dev, Br_dev)

    row = {
        "rep": rep,
        "data_seed": data_seed,

        "fpca_recon_relMSE_vs_clean": fpca_recon_rel_clean,
        "fpca_recon_relMSE_vs_noisy": fpca_recon_rel_noisy,
        "fae_recon_relMSE_vs_clean": fae_recon_rel_clean,
        "fae_recon_relMSE_vs_noisy": fae_recon_rel_noisy,

        "fae_train_mse_final_noisy": _mse(xtr_hat.detach().cpu(), Xtr),
        "fae_val_mse_final_noisy": _mse(xva_hat.detach().cpu(), Xva),

        "fpca_fore_relMSE_vs_clean": fpca_fore_rel_clean,
        "fpca_fore_relMSE_vs_noisy": fpca_fore_rel_noisy,
        "fae_fore_relMSE_vs_clean": fae_fore_rel_clean,
        "fae_fore_relMSE_vs_noisy": fae_fore_rel_noisy,
    }

    row.update({
        "map_mode": "nonlinear",
        "gen_basis_type": sim_cfg.gen_basis_type,
        "gen_n_basis": sim_cfg.gen_n_basis,
        "T": sim_cfg.T,
        "P": sim_cfg.P,
        "d": sim_cfg.d,
        "meas_noise_sd": sim_cfg.meas_noise_sd,
        "fpca_K": run_cfg.fpca_K,
        "horizon": run_cfg.horizon,
        "fae_n_rep": fae_cfg.n_rep,
        "fae_n_basis_project": fae_cfg.n_basis_project,
        "fae_n_basis_revert": fae_cfg.n_basis_revert,
        "latent_nl_scale": sim_cfg.latent_nl_scale,
        "latent_hetero_scale": sim_cfg.latent_hetero_scale,
    })
    return row


# ============================================================
# 8) Many runs
# ============================================================

def run_sim_fpca_fae_many(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 10,
    data_seed_base: int = 30000,
    verbose: bool = False,
) -> pd.DataFrame:
    fixed = build_fixed_decoder(sim_cfg)

    rows: List[Dict[str, float]] = []
    for rep in range(1, n_reps + 1):
        print(f"=== repetition {rep}/{n_reps} ===")
        row = run_one_rep(
            fixed=fixed,
            sim_cfg=sim_cfg,
            run_cfg=run_cfg,
            fae_cfg=fae_cfg,
            rep=rep,
            data_seed_base=data_seed_base,
            verbose=verbose,
        )
        rows.append(row)

        print(
            f"recon(clean): FPCA={row['fpca_recon_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_recon_relMSE_vs_clean']:.4f}"
        )
        print(
            f"fore(clean) : FPCA={row['fpca_fore_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_fore_relMSE_vs_clean']:.4f}"
        )
        print()

    df = pd.DataFrame(rows)
    return df


# ============================================================
# 9) Example
# ============================================================

if __name__ == "__main__":
    sim_cfg = SimCfg(
        seed=123,
        T=500,
        P=100,
        d=5,
        Sigma_scale=0.08,
        target_rho=0.80,
        gen_basis_type="bspline",
        gen_n_basis=12,
        gen_bspline_degree=3,
        map_hidden=(64, 64),
        map_activation="relu",
        map_weight_sd=0.8,
        meas_noise_sd=0.25,      # smaller than before
        latent_nl_scale=0.25,    # NEW
        latent_hetero_scale=0.15 # NEW
    )

    run_cfg = RunCfg(
        horizon=10,              # longer horizon helps expose nonlinear forecast gains
        fpca_K=5,
        fpca_var_ridge=1e-6,
    )

    fae_cfg = FaeCfg(
        seed=743,
        device="cpu",
        n_basis_project=30,
        n_basis_revert=30,
        basis_type_project="bspline",
        basis_type_revert="bspline",
        bspline_degree=3,
        n_rep=8,                 # slightly richer latent space than 5
        init_weight_sd=0.3,
        ae_epochs=1200,
        dyn_epochs=800,
        batch_size=32,
        lr_ae=3e-4,
        lr_dyn=3e-4,
        weight_decay=1e-4,
        split_rate=0.85,
        log_every=200,
        lamb=1e-4,
        dyn_teacher_forcing_noise=0.01,
    )

    df = run_sim_fpca_fae_many(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        data_seed_base=30000,
        verbose=False,
    )

    print("\n========== averages over repetitions ==========")
    avg_cols = [
        "fpca_recon_relMSE_vs_clean",
        "fae_recon_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_clean",
        "fae_fore_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_noisy",
        "fae_fore_relMSE_vs_noisy",
    ]
    print(df[avg_cols].mean())

    print("\n========== full table ==========")
    print(df.head(10))

=== repetition 1/10 ===
recon(clean): FPCA=11.1325 | FAE=0.5581
fore(clean) : FPCA=0.6971 | FAE=0.5383

=== repetition 2/10 ===
recon(clean): FPCA=1.5060 | FAE=0.2353
fore(clean) : FPCA=0.1459 | FAE=0.1356

=== repetition 3/10 ===
recon(clean): FPCA=5.1732 | FAE=0.3277
fore(clean) : FPCA=0.3518 | FAE=0.2938

=== repetition 4/10 ===
recon(clean): FPCA=1.8505 | FAE=0.2970
fore(clean) : FPCA=0.1074 | FAE=0.0852

=== repetition 5/10 ===
recon(clean): FPCA=9.2834 | FAE=0.4640
fore(clean) : FPCA=0.4455 | FAE=0.3595

=== repetition 6/10 ===
recon(clean): FPCA=6.7623 | FAE=0.4334
fore(clean) : FPCA=0.7209 | FAE=0.5841

=== repetition 7/10 ===
recon(clean): FPCA=4.7226 | FAE=0.5237
fore(clean) : FPCA=0.6299 | FAE=0.5479

=== repetition 8/10 ===
recon(clean): FPCA=6.6974 | FAE=0.4374
fore(clean) : FPCA=0.5410 | FAE=0.4307

=== repetition 9/10 ===
recon(clean): FPCA=3.9916 | FAE=0.3124
fore(clean) : FPCA=0.6408 | FAE=0.5409

=== repetition 10/10 ===
recon(clean): FPCA=5.5710 | FAE=0.3121
fore(cle

In [11]:
df.to_excel("sim_results_10runs_MLP.xlsx", index=False)

In [9]:
# ============================================================
# Functional time series simulation:
# FPCA + MLP vs FAE + MLP
# ============================================================

import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import SplineTransformer


# ============================================================
# 0) Repro
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ============================================================
# 1) Configs
# ============================================================

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 500
    P: int = 100
    d: int = 5

    # latent innovation covariance scale
    Sigma_scale: float = 0.08
    target_rho: float = 0.80

    # generator basis
    gen_basis_type: str = "bspline"   # "bspline" or "fourier"
    gen_n_basis: int = 12
    gen_bspline_degree: int = 3

    # nonlinear decoder from latent state -> basis coefficients
    map_hidden: Tuple[int, int] = (64, 64)
    map_activation: str = "relu"
    map_weight_sd: float = 0.8

    # observation noise
    meas_noise_sd: float = 0.25

    # NEW: nonlinear latent dynamics strength
    latent_nl_scale: float = 0.25
    latent_hetero_scale: float = 0.15


@dataclass
class RunCfg:
    horizon: int = 10
    fpca_K: int = 5
    fpca_var_ridge: float = 1e-6


@dataclass
class FaeCfg:
    seed: int = 743
    device: str = "cpu"

    n_basis_project: int = 30
    n_basis_revert: int = 30
    basis_type_project: str = "bspline"
    basis_type_revert: str = "bspline"
    bspline_degree: int = 3

    n_rep: int = 8   # latent representation size
    init_weight_sd: float = 0.3

    ae_epochs: int = 1200
    dyn_epochs: int = 800

    batch_size: int = 32
    lr_ae: float = 3e-4
    lr_dyn: float = 3e-4
    weight_decay: float = 1e-4

    split_rate: float = 0.85
    log_every: int = 100

    # optional coefficient smoothness penalty
    lamb: float = 1e-4

    # multi-step rollout training
    dyn_teacher_forcing_noise: float = 0.01


# ============================================================
# 2) Small utilities
# ============================================================

def _mse(a, b) -> float:
    a = torch.as_tensor(a, dtype=torch.float32)
    b = torch.as_tensor(b, dtype=torch.float32)
    return float(torch.mean((a - b) ** 2).item())

def rel_mse(pred, truth) -> float:
    pred = torch.as_tensor(pred, dtype=torch.float32)
    truth = torch.as_tensor(truth, dtype=torch.float32)
    num = torch.mean((pred - truth) ** 2)
    den = torch.mean(truth ** 2) + 1e-12
    return float((num / den).item())

def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    return X[:-horizon], X[-horizon:]

def activation_from_name(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


# ============================================================
# 3) Basis builders
# ============================================================

def make_grid(P: int) -> np.ndarray:
    return np.linspace(0.0, 1.0, P)

def make_bspline_basis(u: np.ndarray, n_basis: int, degree: int) -> np.ndarray:
    # sklearn SplineTransformer returns basis matrix [P, n_features]
    st = SplineTransformer(
        n_knots=max(n_basis - degree + 1, degree + 1),
        degree=degree,
        include_bias=True
    )
    B = st.fit_transform(u.reshape(-1, 1))
    if B.shape[1] > n_basis:
        B = B[:, :n_basis]
    elif B.shape[1] < n_basis:
        pad = np.zeros((B.shape[0], n_basis - B.shape[1]))
        B = np.hstack([B, pad])
    return B.astype(np.float32)

def make_fourier_basis(u: np.ndarray, n_basis: int) -> np.ndarray:
    cols = [np.ones_like(u)]
    k = 1
    while len(cols) < n_basis:
        cols.append(np.sin(2 * np.pi * k * u))
        if len(cols) < n_basis:
            cols.append(np.cos(2 * np.pi * k * u))
        k += 1
    B = np.column_stack(cols[:n_basis]).astype(np.float32)
    return B

def make_basis(u: np.ndarray, basis_type: str, n_basis: int, degree: int = 3) -> np.ndarray:
    basis_type = basis_type.lower()
    if basis_type == "bspline":
        return make_bspline_basis(u, n_basis, degree)
    if basis_type == "fourier":
        return make_fourier_basis(u, n_basis)
    raise ValueError(f"Unknown basis_type: {basis_type}")


# ============================================================
# 4) Generator pieces
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: Tuple[int, ...], out_dim: int, act: str = "relu"):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(activation_from_name(act))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def stabilize_A(A: np.ndarray, target_rho: float = 0.8) -> np.ndarray:
    eigvals = np.linalg.eigvals(A)
    rho = float(np.max(np.abs(eigvals)))
    if rho < 1e-12:
        return A
    return A * (target_rho / rho)


def generate_latent_nonlinear_ar1(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    target_rho: float = 0.8,
    nl_scale: float = 0.25,
    hetero_scale: float = 0.15,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Mildly nonlinear latent dynamics.
    Still close to AR(1), but no longer ideal for linear FPCA forecasting.

    z_t = A z_{t-1}
          + nonlinear_drift(z_{t-1})
          + state-dependent noise
    """
    rng = np.random.default_rng(seed)

    A = rng.normal(size=(d, d)) * 0.20
    A = stabilize_A(A, target_rho=target_rho)

    Z = np.zeros((T, d), dtype=np.float32)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma).astype(np.float32)

    for t in range(1, T):
        zprev = Z[t - 1].copy()
        Az = A @ zprev

        # Local nonlinear drift on first coordinates
        drift = np.zeros(d, dtype=np.float32)
        if d >= 1:
            drift[0] += nl_scale * (zprev[0] ** 2 - 0.75)
        if d >= 2:
            drift[1] += 0.8 * nl_scale * np.sin(1.5 * zprev[1])
        if d >= 3:
            drift[2] += 0.6 * nl_scale * np.tanh(zprev[0] * zprev[2])
        if d >= 4:
            drift[3] += 0.5 * nl_scale * (zprev[1] * zprev[3])

        scale = 1.0 + hetero_scale * abs(float(zprev[0]))
        eps = rng.multivariate_normal(np.zeros(d), (scale ** 2) * Sigma).astype(np.float32)

        Z[t] = Az + drift + eps

    return Z, A


def build_fixed_decoder(sim_cfg: SimCfg) -> Dict[str, object]:
    set_seed(sim_cfg.seed)

    u = make_grid(sim_cfg.P)
    Bgen_np = make_basis(
        u=u,
        basis_type=sim_cfg.gen_basis_type,
        n_basis=sim_cfg.gen_n_basis,
        degree=sim_cfg.gen_bspline_degree
    )

    mapper = MLP(
        in_dim=sim_cfg.d,
        hidden=sim_cfg.map_hidden,
        out_dim=sim_cfg.gen_n_basis,
        act=sim_cfg.map_activation,
    )

    for m in mapper.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)

    return {
        "u": torch.tensor(u, dtype=torch.float32),
        "tpts": torch.tensor(u, dtype=torch.float32),
        "Bgen": torch.tensor(Bgen_np, dtype=torch.float32),
        "mapper": mapper.eval(),
    }


def simulate_functional_ts(sim_cfg: SimCfg) -> Dict[str, torch.Tensor]:
    fixed = build_fixed_decoder(sim_cfg)
    return _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=sim_cfg.seed + 1000)


def _simulate_new_dataset_same_decoder(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    data_seed: int,
) -> Dict[str, torch.Tensor]:
    Sigma = (sim_cfg.Sigma_scale ** 2) * np.eye(sim_cfg.d, dtype=np.float32)

    Z, A_true = generate_latent_nonlinear_ar1(
        T=sim_cfg.T,
        d=sim_cfg.d,
        Sigma=Sigma,
        seed=data_seed,
        target_rho=sim_cfg.target_rho,
        nl_scale=sim_cfg.latent_nl_scale,
        hetero_scale=sim_cfg.latent_hetero_scale,
    )

    Z_t = torch.tensor(Z, dtype=torch.float32)

    mapper: nn.Module = fixed["mapper"]
    Bgen: torch.Tensor = fixed["Bgen"]

    with torch.no_grad():
        coef = mapper(Z_t)                   # [T, M]
        X_clean = coef @ Bgen.T              # [T, P]
        X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)

    return {
        "u": fixed["u"],
        "tpts": fixed["tpts"],
        "Z": Z_t,
        "A_true": torch.tensor(A_true, dtype=torch.float32),
        "X_clean": X_clean,
        "X_noisy": X_noisy,
    }


# ============================================================
# 5) FPCA baseline
# ============================================================

def fit_var1(Y: np.ndarray, ridge: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    """
    Y_t = b + A Y_{t-1}
    """
    X = Y[:-1]
    Z = Y[1:]
    X1 = np.hstack([np.ones((X.shape[0], 1)), X])  # [n, 1+p]

    XtX = X1.T @ X1
    reg = ridge * np.eye(X1.shape[1], dtype=np.float64)
    B = np.linalg.solve(XtX + reg, X1.T @ Z)       # [1+p, p]

    b = B[0]
    A = B[1:].T
    return b.astype(np.float32), A.astype(np.float32)

def forecast_var1(y_last: np.ndarray, b: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    out = []
    cur = y_last.astype(np.float32).copy()
    for _ in range(steps):
        cur = b + A @ cur
        out.append(cur.copy())
    return np.stack(out, axis=0)

def fpca_reconstruct_uncentered_from_trainfit(
    X_train: torch.Tensor,
    X_eval: torch.Tensor,
    K: int,
) -> Tuple[torch.Tensor, PCA, np.ndarray]:
    pca = PCA(n_components=K, svd_solver="full")
    scores_train = pca.fit_transform(X_train.numpy())
    scores_eval = pca.transform(X_eval.numpy())
    Xhat = pca.inverse_transform(scores_eval)
    return torch.tensor(Xhat, dtype=torch.float32), pca, scores_train

def fpca_var_forecast_uncentered(
    X_train: torch.Tensor,
    K: int,
    steps: int,
    var_ridge: float = 1e-6
) -> torch.Tensor:
    pca = PCA(n_components=K, svd_solver="full")
    scores = pca.fit_transform(X_train.numpy())
    b, A = fit_var1(scores, ridge=var_ridge)
    Sf = forecast_var1(scores[-1], b, A, steps)
    Xf = pca.inverse_transform(Sf)
    return torch.tensor(Xf, dtype=torch.float32)


def train_fpca_dynamics_mlp(
    Xn_train: torch.Tensor,
    K: int,
    hidden=(64, 64),
    epochs=800,
    lr=3e-4,
    batch_size=32,
):
    device = Xn_train.device

    # PCA
    pca = PCA(n_components=K, svd_solver="full")
    scores = torch.tensor(pca.fit_transform(Xn_train.numpy()), dtype=torch.float32, device=device)

    H_in = scores[:-1]
    H_out = scores[1:]

    # MLP (same style as FAE)
    dyn = LatentDynamicsMLP(n_rep=K, hidden=hidden).to(device)
    init_small_weights(dyn, 0.3)

    opt = optim.Adam(dyn.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for ep in range(epochs):
        perm = torch.randperm(H_in.shape[0], device=device)

        for i in range(0, H_in.shape[0], batch_size):
            idx = perm[i:i + batch_size]
            pred = dyn(H_in[idx])
            loss = loss_fn(pred, H_out[idx])

            opt.zero_grad()
            loss.backward()
            opt.step()

    return dyn, pca

@torch.no_grad()
def fpca_mlp_forecast(
    Xn_train: torch.Tensor,
    dyn,
    pca,
    steps: int,
):
    device = Xn_train.device

    scores = torch.tensor(pca.transform(Xn_train.numpy()), dtype=torch.float32, device=device)

    h = scores[-1:].clone()
    preds = []

    for _ in range(steps):
        h = dyn(h)
        xhat = torch.tensor(pca.inverse_transform(h.cpu().numpy()), dtype=torch.float32)
        preds.append(xhat.squeeze(0))

    return torch.stack(preds, dim=0)

# ============================================================
# 6) FAE model
# ============================================================

class FAEVanilla(nn.Module):
    """
    Project x(t) onto basis Bp -> coefficients -> encoder -> latent rep
    latent rep -> decoder -> coefficients on Br -> reconstruct curve
    """
    def __init__(
        self,
        P: int,
        n_basis_project: int,
        n_basis_revert: int,
        n_rep: int,
        hidden: Tuple[int, int] = (64, 64),
    ):
        super().__init__()

        self.enc = nn.Sequential(
            nn.Linear(n_basis_project, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

        self.dec = nn.Sequential(
            nn.Linear(n_rep, hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], n_basis_revert),
        )

        self.P = P
        self.n_basis_project = n_basis_project
        self.n_basis_revert = n_basis_revert
        self.n_rep = n_rep

    def encode_from_curve(self, X: torch.Tensor, Bp: torch.Tensor) -> torch.Tensor:
        # least-squares projection onto basis
        # coef = argmin ||X - coef Bp^T||^2
        G = Bp.T @ Bp + 1e-6 * torch.eye(Bp.shape[1], device=Bp.device)
        coef = torch.linalg.solve(G, Bp.T @ X.T).T
        H = self.enc(coef)
        return H

    def decode_from_rep(self, H: torch.Tensor, Br: torch.Tensor) -> torch.Tensor:
        coef = self.dec(H)
        Xhat = coef @ Br.T
        return Xhat

    def forward(self, X: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor):
        H = self.encode_from_curve(X, Bp)
        coef_r = self.dec(H)
        Xhat = coef_r @ Br.T
        return Xhat, H, coef_r


class LatentDynamicsMLP(nn.Module):
    """
    One-step latent transition: h_{t+1} = g(h_t)
    """
    def __init__(self, n_rep: int, hidden: Tuple[int, int] = (64, 64)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_rep, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

    def forward(self, h):
        return self.net(h)


def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    if coef.shape[1] <= 1:
        return torch.tensor(0.0, device=coef.device)
    d1 = coef[:, 1:] - coef[:, :-1]
    return torch.mean(d1 ** 2)


def init_small_weights(model: nn.Module, sd: float) -> None:
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)


def train_fae_on_noisy_train(
    Xn_train: torch.Tensor,
    tpts: torch.Tensor,
    cfg: FaeCfg,
) -> Tuple[FAEVanilla, torch.Tensor, torch.Tensor, torch.Tensor]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    P = Xn_train.shape[1]
    u = tpts.numpy()

    Bp_np = make_basis(u, cfg.basis_type_project, cfg.n_basis_project, cfg.bspline_degree)
    Br_np = make_basis(u, cfg.basis_type_revert, cfg.n_basis_revert, cfg.bspline_degree)

    Bp = torch.tensor(Bp_np, dtype=torch.float32, device=device)
    Br = torch.tensor(Br_np, dtype=torch.float32, device=device)

    model = FAEVanilla(
        P=P,
        n_basis_project=cfg.n_basis_project,
        n_basis_revert=cfg.n_basis_revert,
        n_rep=cfg.n_rep,
        hidden=(64, 64),
    ).to(device)

    init_small_weights(model, cfg.init_weight_sd)

    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=cfg.split_rate,
        random_state=cfg.seed,
        shuffle=True
    )

    Xtr = Xn_train[idx_tr].to(device).float()
    Xva = Xn_train[idx_va].to(device).float()

    opt = optim.Adam(model.parameters(), lr=cfg.lr_ae, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.ae_epochs + 1):
        model.train()
        perm = torch.randperm(Xtr.shape[0], device=device)

        for i in range(0, Xtr.shape[0], cfg.batch_size):
            batch = Xtr[perm[i:i + cfg.batch_size]]
            opt.zero_grad()
            xhat, _, coef = model(batch, Bp, Br)
            loss = loss_fn(xhat, batch) + cfg.lamb * diff_penalty(coef)
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            model.eval()
            with torch.no_grad():
                tr_hat, _, _ = model(Xtr, Bp, Br)
                va_hat, _, _ = model(Xva, Bp, Br)
                tr = loss_fn(tr_hat, Xtr).item()
                va = loss_fn(va_hat, Xva).item()
            print(f"[AE ] ep {ep:4d} | train_mse={tr:.6e} | val_mse={va:.6e}")

    return model.eval(), tpts.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()


@torch.no_grad()
def fae_reconstruct(
    model: FAEVanilla,
    X: torch.Tensor,
    Bp: torch.Tensor,
    Br: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    device = next(model.parameters()).device
    X = X.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)
    Xhat, H, _ = model(X, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu()


def train_fae_dynamics_head(
    model: FAEVanilla,
    Xn_train: torch.Tensor,
    Bp: torch.Tensor,
    cfg: FaeCfg,
) -> LatentDynamicsMLP:
    """
    Train nonlinear one-step latent transition on encoded FAE states.
    """
    set_seed(cfg.seed + 17)
    device = next(model.parameters()).device
    Bp = Bp.to(device)

    model.eval()
    with torch.no_grad():
        H = model.encode_from_curve(Xn_train.to(device).float(), Bp)

    H_in = H[:-1]
    H_out = H[1:]

    dyn = LatentDynamicsMLP(n_rep=cfg.n_rep, hidden=(64, 64)).to(device)
    init_small_weights(dyn, cfg.init_weight_sd)

    opt = optim.Adam(dyn.parameters(), lr=cfg.lr_dyn, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.dyn_epochs + 1):
        dyn.train()
        perm = torch.randperm(H_in.shape[0], device=device)

        for i in range(0, H_in.shape[0], cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            hin = H_in[idx]
            hout = H_out[idx]

            if cfg.dyn_teacher_forcing_noise > 0:
                hin = hin + cfg.dyn_teacher_forcing_noise * torch.randn_like(hin)

            pred = dyn(hin)
            loss = loss_fn(pred, hout)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            dyn.eval()
            with torch.no_grad():
                pred = dyn(H_in)
                tr = loss_fn(pred, H_out).item()
            print(f"[DYN] ep {ep:4d} | one_step_latent_mse={tr:.6e}")

    return dyn.eval()


@torch.no_grad()
def fae_nonlinear_forecast(
    model: FAEVanilla,
    dyn: LatentDynamicsMLP,
    Xn_train: torch.Tensor,
    steps: int,
    Bp: torch.Tensor,
    Br: torch.Tensor,
) -> torch.Tensor:
    device = next(model.parameters()).device
    Bp = Bp.to(device)
    Br = Br.to(device)

    H = model.encode_from_curve(Xn_train.to(device).float(), Bp)
    h = H[-1:].clone()

    preds = []
    for _ in range(steps):
        h = dyn(h)
        xhat = model.decode_from_rep(h, Br)
        preds.append(xhat.squeeze(0).detach().cpu())

    return torch.stack(preds, dim=0)


# ============================================================
# 7) One run
# ============================================================

def run_one_rep(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    rep: int,
    data_seed_base: int,
    verbose: bool = False,
) -> Dict[str, float]:
    data_seed = data_seed_base + rep
    sim = _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=data_seed)

    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    # ---------- FPCA reconstruction ----------
    X_fpca_recon_train, _, _ = fpca_reconstruct_uncentered_from_trainfit(
        X_train=Xn_train,
        X_eval=Xn_train,
        K=run_cfg.fpca_K,
    )

    # ---------- FAE reconstruction ----------
    old_log = fae_cfg.log_every
    if not verbose:
        fae_cfg.log_every = 0

    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_noisy_train(Xn_train, tpts, fae_cfg)
    Bp_dev = Bp_fae.to(fae_cfg.device)
    Br_dev = Br_fae.to(fae_cfg.device)

    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, Bp_dev, Br_dev)

    # ---------- Forecast ----------
    dyn_fpca, pca_model = train_fpca_dynamics_mlp(
        Xn_train,
        K=run_cfg.fpca_K,
        epochs=fae_cfg.dyn_epochs,   # reuse same training budget
    )

    X_fpca_fore = fpca_mlp_forecast(
        Xn_train,
        dyn_fpca,
        pca_model,
        steps=run_cfg.horizon,
    )

    dyn_head = train_fae_dynamics_head(model_fae, Xn_train, Bp_dev, fae_cfg)
    X_fae_fore = fae_nonlinear_forecast(
        model=model_fae,
        dyn=dyn_head,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        Bp=Bp_dev,
        Br=Br_dev,
    )

    fae_cfg.log_every = old_log

    # ---------- Metrics ----------
    # Reconstruction
    fpca_recon_rel_clean = rel_mse(X_fpca_recon_train, Xc_train)
    fpca_recon_rel_noisy = rel_mse(X_fpca_recon_train, Xn_train)
    fae_recon_rel_clean = rel_mse(X_fae_recon_train, Xc_train)
    fae_recon_rel_noisy = rel_mse(X_fae_recon_train, Xn_train)

    # Forecast
    fpca_fore_rel_clean = rel_mse(X_fpca_fore, Xc_test)
    fpca_fore_rel_noisy = rel_mse(X_fpca_fore, Xn_test)
    fae_fore_rel_clean = rel_mse(X_fae_fore, Xc_test)
    fae_fore_rel_noisy = rel_mse(X_fae_fore, Xn_test)

    # Final AE train/val MSE
    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=fae_cfg.split_rate,
        random_state=fae_cfg.seed,
        shuffle=True
    )
    Xtr = Xn_train[idx_tr].float()
    Xva = Xn_train[idx_va].float()

    model_fae.eval()
    with torch.no_grad():
        xtr_hat, _, _ = model_fae(Xtr.to(fae_cfg.device), Bp_dev, Br_dev)
        xva_hat, _, _ = model_fae(Xva.to(fae_cfg.device), Bp_dev, Br_dev)

    row = {
        "rep": rep,
        "data_seed": data_seed,

        "fpca_recon_relMSE_vs_clean": fpca_recon_rel_clean,
        "fpca_recon_relMSE_vs_noisy": fpca_recon_rel_noisy,
        "fae_recon_relMSE_vs_clean": fae_recon_rel_clean,
        "fae_recon_relMSE_vs_noisy": fae_recon_rel_noisy,

        "fae_train_mse_final_noisy": _mse(xtr_hat.detach().cpu(), Xtr),
        "fae_val_mse_final_noisy": _mse(xva_hat.detach().cpu(), Xva),

        "fpca_fore_relMSE_vs_clean": fpca_fore_rel_clean,
        "fpca_fore_relMSE_vs_noisy": fpca_fore_rel_noisy,
        "fae_fore_relMSE_vs_clean": fae_fore_rel_clean,
        "fae_fore_relMSE_vs_noisy": fae_fore_rel_noisy,
    }

    row.update({
        "map_mode": "nonlinear",
        "gen_basis_type": sim_cfg.gen_basis_type,
        "gen_n_basis": sim_cfg.gen_n_basis,
        "T": sim_cfg.T,
        "P": sim_cfg.P,
        "d": sim_cfg.d,
        "meas_noise_sd": sim_cfg.meas_noise_sd,
        "fpca_K": run_cfg.fpca_K,
        "horizon": run_cfg.horizon,
        "fae_n_rep": fae_cfg.n_rep,
        "fae_n_basis_project": fae_cfg.n_basis_project,
        "fae_n_basis_revert": fae_cfg.n_basis_revert,
        "latent_nl_scale": sim_cfg.latent_nl_scale,
        "latent_hetero_scale": sim_cfg.latent_hetero_scale,
    })
    return row


# ============================================================
# 8) Many runs
# ============================================================

def run_sim_fpca_fae_many(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 10,
    data_seed_base: int = 30000,
    verbose: bool = False,
) -> pd.DataFrame:
    fixed = build_fixed_decoder(sim_cfg)

    rows: List[Dict[str, float]] = []
    for rep in range(1, n_reps + 1):
        print(f"=== repetition {rep}/{n_reps} ===")
        row = run_one_rep(
            fixed=fixed,
            sim_cfg=sim_cfg,
            run_cfg=run_cfg,
            fae_cfg=fae_cfg,
            rep=rep,
            data_seed_base=data_seed_base,
            verbose=verbose,
        )
        rows.append(row)

        print(
            f"recon(clean): FPCA={row['fpca_recon_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_recon_relMSE_vs_clean']:.4f}"
        )
        print(
            f"fore(clean) : FPCA={row['fpca_fore_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_fore_relMSE_vs_clean']:.4f}"
        )
        print()

    df = pd.DataFrame(rows)
    return df


# ============================================================
# 9) Example
# ============================================================

if __name__ == "__main__":
    sim_cfg = SimCfg(
        seed=123,
        T=500,
        P=100,
        d=5,
        Sigma_scale=0.08,
        target_rho=0.80,
        gen_basis_type="bspline",
        gen_n_basis=12,
        gen_bspline_degree=3,
        map_hidden=(64, 64),
        map_activation="relu",
        map_weight_sd=0.8,
        meas_noise_sd=0.25,      # smaller than before
        latent_nl_scale=0.25,    # NEW
        latent_hetero_scale=0.15 # NEW
    )

    run_cfg = RunCfg(
        horizon=10,              # longer horizon helps expose nonlinear forecast gains
        fpca_K=5,
        fpca_var_ridge=1e-6,
    )

    fae_cfg = FaeCfg(
        seed=743,
        device="cpu",
        n_basis_project=30,
        n_basis_revert=30,
        basis_type_project="bspline",
        basis_type_revert="bspline",
        bspline_degree=3,
        n_rep=8,                 # slightly richer latent space than 5
        init_weight_sd=0.3,
        ae_epochs=1200,
        dyn_epochs=800,
        batch_size=32,
        lr_ae=3e-4,
        lr_dyn=3e-4,
        weight_decay=1e-4,
        split_rate=0.85,
        log_every=200,
        lamb=1e-4,
        dyn_teacher_forcing_noise=0.01,
    )

    df = run_sim_fpca_fae_many(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        data_seed_base=30000,
        verbose=False,
    )

    print("\n========== averages over repetitions ==========")
    avg_cols = [
        "fpca_recon_relMSE_vs_clean",
        "fae_recon_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_clean",
        "fae_fore_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_noisy",
        "fae_fore_relMSE_vs_noisy",
    ]
    print(df[avg_cols].mean())

    print("\n========== full table ==========")
    print(df.head(10))

=== repetition 1/10 ===
recon(clean): FPCA=11.1325 | FAE=0.5581
fore(clean) : FPCA=2.4511 | FAE=0.5383

=== repetition 2/10 ===
recon(clean): FPCA=1.5060 | FAE=0.2353
fore(clean) : FPCA=0.1994 | FAE=0.1356

=== repetition 3/10 ===
recon(clean): FPCA=5.1732 | FAE=0.3277
fore(clean) : FPCA=1.1778 | FAE=0.2938

=== repetition 4/10 ===
recon(clean): FPCA=1.8505 | FAE=0.2970
fore(clean) : FPCA=0.2454 | FAE=0.0852

=== repetition 5/10 ===
recon(clean): FPCA=9.2834 | FAE=0.4640
fore(clean) : FPCA=1.5014 | FAE=0.3595

=== repetition 6/10 ===
recon(clean): FPCA=6.7623 | FAE=0.4334
fore(clean) : FPCA=2.6092 | FAE=0.5841

=== repetition 7/10 ===
recon(clean): FPCA=4.7226 | FAE=0.5237
fore(clean) : FPCA=1.1829 | FAE=0.5479

=== repetition 8/10 ===
recon(clean): FPCA=6.6974 | FAE=0.4374
fore(clean) : FPCA=1.5703 | FAE=0.4307

=== repetition 9/10 ===
recon(clean): FPCA=3.9916 | FAE=0.3124
fore(clean) : FPCA=1.7006 | FAE=0.5409

=== repetition 10/10 ===
recon(clean): FPCA=5.5710 | FAE=0.3121
fore(cle

In [ ]:
# ============================================================
# Functional time series simulation:
# FPCA vs FAE + kernel regression forecast head
# ============================================================

import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import SplineTransformer


# ============================================================
# 0) Repro
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ============================================================
# 1) Configs
# ============================================================

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 500
    P: int = 100
    d: int = 5

    # latent innovation covariance scale
    Sigma_scale: float = 0.08
    target_rho: float = 0.80

    # generator basis
    gen_basis_type: str = "bspline"   # "bspline" or "fourier"
    gen_n_basis: int = 12
    gen_bspline_degree: int = 3

    # nonlinear decoder from latent state -> basis coefficients
    map_hidden: Tuple[int, int] = (64, 64)
    map_activation: str = "relu"
    map_weight_sd: float = 0.8

    # observation noise
    meas_noise_sd: float = 0.25

    # NEW: nonlinear latent dynamics strength
    latent_nl_scale: float = 0.25
    latent_hetero_scale: float = 0.15


@dataclass
class RunCfg:
    horizon: int = 10
    fpca_K: int = 5
    fpca_var_ridge: float = 1e-6


@dataclass
class FaeCfg:
    seed: int = 743
    device: str = "cpu"

    n_basis_project: int = 30
    n_basis_revert: int = 30
    basis_type_project: str = "bspline"
    basis_type_revert: str = "bspline"
    bspline_degree: int = 3

    n_rep: int = 8   # latent representation size
    init_weight_sd: float = 0.3

    ae_epochs: int = 1200
    dyn_epochs: int = 800

    batch_size: int = 32
    lr_ae: float = 3e-4
    lr_dyn: float = 3e-4
    weight_decay: float = 1e-4

    split_rate: float = 0.85
    log_every: int = 100

    # optional coefficient smoothness penalty
    lamb: float = 1e-4

    # multi-step rollout training
    dyn_teacher_forcing_noise: float = 0.01


# ============================================================
# 2) Small utilities
# ============================================================

def _mse(a, b) -> float:
    a = torch.as_tensor(a, dtype=torch.float32)
    b = torch.as_tensor(b, dtype=torch.float32)
    return float(torch.mean((a - b) ** 2).item())

def rel_mse(pred, truth) -> float:
    pred = torch.as_tensor(pred, dtype=torch.float32)
    truth = torch.as_tensor(truth, dtype=torch.float32)
    num = torch.mean((pred - truth) ** 2)
    den = torch.mean(truth ** 2) + 1e-12
    return float((num / den).item())

def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    return X[:-horizon], X[-horizon:]

def activation_from_name(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


# ============================================================
# 3) Basis builders
# ============================================================

def make_grid(P: int) -> np.ndarray:
    return np.linspace(0.0, 1.0, P)

def make_bspline_basis(u: np.ndarray, n_basis: int, degree: int) -> np.ndarray:
    # sklearn SplineTransformer returns basis matrix [P, n_features]
    st = SplineTransformer(
        n_knots=max(n_basis - degree + 1, degree + 1),
        degree=degree,
        include_bias=True
    )
    B = st.fit_transform(u.reshape(-1, 1))
    if B.shape[1] > n_basis:
        B = B[:, :n_basis]
    elif B.shape[1] < n_basis:
        pad = np.zeros((B.shape[0], n_basis - B.shape[1]))
        B = np.hstack([B, pad])
    return B.astype(np.float32)

def make_fourier_basis(u: np.ndarray, n_basis: int) -> np.ndarray:
    cols = [np.ones_like(u)]
    k = 1
    while len(cols) < n_basis:
        cols.append(np.sin(2 * np.pi * k * u))
        if len(cols) < n_basis:
            cols.append(np.cos(2 * np.pi * k * u))
        k += 1
    B = np.column_stack(cols[:n_basis]).astype(np.float32)
    return B

def make_basis(u: np.ndarray, basis_type: str, n_basis: int, degree: int = 3) -> np.ndarray:
    basis_type = basis_type.lower()
    if basis_type == "bspline":
        return make_bspline_basis(u, n_basis, degree)
    if basis_type == "fourier":
        return make_fourier_basis(u, n_basis)
    raise ValueError(f"Unknown basis_type: {basis_type}")


# ============================================================
# 4) Generator pieces
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: Tuple[int, ...], out_dim: int, act: str = "relu"):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(activation_from_name(act))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def stabilize_A(A: np.ndarray, target_rho: float = 0.8) -> np.ndarray:
    eigvals = np.linalg.eigvals(A)
    rho = float(np.max(np.abs(eigvals)))
    if rho < 1e-12:
        return A
    return A * (target_rho / rho)


def generate_latent_nonlinear_ar1(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    target_rho: float = 0.8,
    nl_scale: float = 0.25,
    hetero_scale: float = 0.15,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Mildly nonlinear latent dynamics.
    Still close to AR(1), but no longer ideal for linear FPCA forecasting.

    z_t = A z_{t-1}
          + nonlinear_drift(z_{t-1})
          + state-dependent noise
    """
    rng = np.random.default_rng(seed)

    A = rng.normal(size=(d, d)) * 0.20
    A = stabilize_A(A, target_rho=target_rho)

    Z = np.zeros((T, d), dtype=np.float32)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma).astype(np.float32)

    for t in range(1, T):
        zprev = Z[t - 1].copy()
        Az = A @ zprev

        # Local nonlinear drift on first coordinates
        drift = np.zeros(d, dtype=np.float32)
        if d >= 1:
            drift[0] += nl_scale * (zprev[0] ** 2 - 0.75)
        if d >= 2:
            drift[1] += 0.8 * nl_scale * np.sin(1.5 * zprev[1])
        if d >= 3:
            drift[2] += 0.6 * nl_scale * np.tanh(zprev[0] * zprev[2])
        if d >= 4:
            drift[3] += 0.5 * nl_scale * (zprev[1] * zprev[3])

        scale = 1.0 + hetero_scale * abs(float(zprev[0]))
        eps = rng.multivariate_normal(np.zeros(d), (scale ** 2) * Sigma).astype(np.float32)

        Z[t] = Az + drift + eps

    return Z, A


def build_fixed_decoder(sim_cfg: SimCfg) -> Dict[str, object]:
    set_seed(sim_cfg.seed)

    u = make_grid(sim_cfg.P)
    Bgen_np = make_basis(
        u=u,
        basis_type=sim_cfg.gen_basis_type,
        n_basis=sim_cfg.gen_n_basis,
        degree=sim_cfg.gen_bspline_degree
    )

    mapper = MLP(
        in_dim=sim_cfg.d,
        hidden=sim_cfg.map_hidden,
        out_dim=sim_cfg.gen_n_basis,
        act=sim_cfg.map_activation,
    )

    for m in mapper.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)

    return {
        "u": torch.tensor(u, dtype=torch.float32),
        "tpts": torch.tensor(u, dtype=torch.float32),
        "Bgen": torch.tensor(Bgen_np, dtype=torch.float32),
        "mapper": mapper.eval(),
    }


def simulate_functional_ts(sim_cfg: SimCfg) -> Dict[str, torch.Tensor]:
    fixed = build_fixed_decoder(sim_cfg)
    return _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=sim_cfg.seed + 1000)


def _simulate_new_dataset_same_decoder(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    data_seed: int,
) -> Dict[str, torch.Tensor]:
    Sigma = (sim_cfg.Sigma_scale ** 2) * np.eye(sim_cfg.d, dtype=np.float32)

    Z, A_true = generate_latent_nonlinear_ar1(
        T=sim_cfg.T,
        d=sim_cfg.d,
        Sigma=Sigma,
        seed=data_seed,
        target_rho=sim_cfg.target_rho,
        nl_scale=sim_cfg.latent_nl_scale,
        hetero_scale=sim_cfg.latent_hetero_scale,
    )

    Z_t = torch.tensor(Z, dtype=torch.float32)

    mapper: nn.Module = fixed["mapper"]
    Bgen: torch.Tensor = fixed["Bgen"]

    with torch.no_grad():
        coef = mapper(Z_t)                   # [T, M]
        X_clean = coef @ Bgen.T              # [T, P]
        X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)

    return {
        "u": fixed["u"],
        "tpts": fixed["tpts"],
        "Z": Z_t,
        "A_true": torch.tensor(A_true, dtype=torch.float32),
        "X_clean": X_clean,
        "X_noisy": X_noisy,
    }


# ============================================================
# 5) FPCA baseline
# ============================================================

def fit_var1(Y: np.ndarray, ridge: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    """
    Y_t = b + A Y_{t-1}
    """
    X = Y[:-1]
    Z = Y[1:]
    X1 = np.hstack([np.ones((X.shape[0], 1)), X])  # [n, 1+p]

    XtX = X1.T @ X1
    reg = ridge * np.eye(X1.shape[1], dtype=np.float64)
    B = np.linalg.solve(XtX + reg, X1.T @ Z)       # [1+p, p]

    b = B[0]
    A = B[1:].T
    return b.astype(np.float32), A.astype(np.float32)

def forecast_var1(y_last: np.ndarray, b: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    out = []
    cur = y_last.astype(np.float32).copy()
    for _ in range(steps):
        cur = b + A @ cur
        out.append(cur.copy())
    return np.stack(out, axis=0)

def fpca_reconstruct_uncentered_from_trainfit(
    X_train: torch.Tensor,
    X_eval: torch.Tensor,
    K: int,
) -> Tuple[torch.Tensor, PCA, np.ndarray]:
    pca = PCA(n_components=K, svd_solver="full")
    scores_train = pca.fit_transform(X_train.numpy())
    scores_eval = pca.transform(X_eval.numpy())
    Xhat = pca.inverse_transform(scores_eval)
    return torch.tensor(Xhat, dtype=torch.float32), pca, scores_train

def fpca_var_forecast_uncentered(
    X_train: torch.Tensor,
    K: int,
    steps: int,
    var_ridge: float = 1e-6
) -> torch.Tensor:
    pca = PCA(n_components=K, svd_solver="full")
    scores = pca.fit_transform(X_train.numpy())
    b, A = fit_var1(scores, ridge=var_ridge)
    Sf = forecast_var1(scores[-1], b, A, steps)
    Xf = pca.inverse_transform(Sf)
    return torch.tensor(Xf, dtype=torch.float32)

def kernel_forecast(
    H_train: torch.Tensor,
    steps: int,
    bandwidth: float = 1.0,
):
    """
    Nonparametric kernel regression:
    H_{t+1} = weighted average of next states
    """

    device = H_train.device

    H_in = H_train[:-1]   # [T-1, d]
    H_out = H_train[1:]   # [T-1, d]

    h = H_train[-1].clone()
    preds = []

    for _ in range(steps):
        # compute distances
        diff = H_in - h.unsqueeze(0)              # [T-1, d]
        dist2 = torch.sum(diff**2, dim=1)         # [T-1]

        # kernel weights (Gaussian)
        weights = torch.exp(-dist2 / (2 * bandwidth**2))
        weights = weights / (weights.sum() + 1e-8)

        # weighted average of next states
        h = torch.sum(weights.unsqueeze(1) * H_out, dim=0)

        preds.append(h.clone())

    return torch.stack(preds, dim=0)

# ============================================================
# 6) FAE model
# ============================================================

class FAEVanilla(nn.Module):
    """
    Project x(t) onto basis Bp -> coefficients -> encoder -> latent rep
    latent rep -> decoder -> coefficients on Br -> reconstruct curve
    """
    def __init__(
        self,
        P: int,
        n_basis_project: int,
        n_basis_revert: int,
        n_rep: int,
        hidden: Tuple[int, int] = (64, 64),
    ):
        super().__init__()

        self.enc = nn.Sequential(
            nn.Linear(n_basis_project, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

        self.dec = nn.Sequential(
            nn.Linear(n_rep, hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], n_basis_revert),
        )

        self.P = P
        self.n_basis_project = n_basis_project
        self.n_basis_revert = n_basis_revert
        self.n_rep = n_rep

    def encode_from_curve(self, X: torch.Tensor, Bp: torch.Tensor) -> torch.Tensor:
        # least-squares projection onto basis
        # coef = argmin ||X - coef Bp^T||^2
        G = Bp.T @ Bp + 1e-6 * torch.eye(Bp.shape[1], device=Bp.device)
        coef = torch.linalg.solve(G, Bp.T @ X.T).T
        H = self.enc(coef)
        return H

    def decode_from_rep(self, H: torch.Tensor, Br: torch.Tensor) -> torch.Tensor:
        coef = self.dec(H)
        Xhat = coef @ Br.T
        return Xhat

    def forward(self, X: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor):
        H = self.encode_from_curve(X, Bp)
        coef_r = self.dec(H)
        Xhat = coef_r @ Br.T
        return Xhat, H, coef_r


class LatentDynamicsMLP(nn.Module):
    """
    One-step latent transition: h_{t+1} = g(h_t)
    """
    def __init__(self, n_rep: int, hidden: Tuple[int, int] = (64, 64)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_rep, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

    def forward(self, h):
        return self.net(h)


def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    if coef.shape[1] <= 1:
        return torch.tensor(0.0, device=coef.device)
    d1 = coef[:, 1:] - coef[:, :-1]
    return torch.mean(d1 ** 2)


def init_small_weights(model: nn.Module, sd: float) -> None:
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)


def train_fae_on_noisy_train(
    Xn_train: torch.Tensor,
    tpts: torch.Tensor,
    cfg: FaeCfg,
) -> Tuple[FAEVanilla, torch.Tensor, torch.Tensor, torch.Tensor]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    P = Xn_train.shape[1]
    u = tpts.numpy()

    Bp_np = make_basis(u, cfg.basis_type_project, cfg.n_basis_project, cfg.bspline_degree)
    Br_np = make_basis(u, cfg.basis_type_revert, cfg.n_basis_revert, cfg.bspline_degree)

    Bp = torch.tensor(Bp_np, dtype=torch.float32, device=device)
    Br = torch.tensor(Br_np, dtype=torch.float32, device=device)

    model = FAEVanilla(
        P=P,
        n_basis_project=cfg.n_basis_project,
        n_basis_revert=cfg.n_basis_revert,
        n_rep=cfg.n_rep,
        hidden=(64, 64),
    ).to(device)

    init_small_weights(model, cfg.init_weight_sd)

    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=cfg.split_rate,
        random_state=cfg.seed,
        shuffle=True
    )

    Xtr = Xn_train[idx_tr].to(device).float()
    Xva = Xn_train[idx_va].to(device).float()

    opt = optim.Adam(model.parameters(), lr=cfg.lr_ae, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.ae_epochs + 1):
        model.train()
        perm = torch.randperm(Xtr.shape[0], device=device)

        for i in range(0, Xtr.shape[0], cfg.batch_size):
            batch = Xtr[perm[i:i + cfg.batch_size]]
            opt.zero_grad()
            xhat, _, coef = model(batch, Bp, Br)
            loss = loss_fn(xhat, batch) + cfg.lamb * diff_penalty(coef)
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            model.eval()
            with torch.no_grad():
                tr_hat, _, _ = model(Xtr, Bp, Br)
                va_hat, _, _ = model(Xva, Bp, Br)
                tr = loss_fn(tr_hat, Xtr).item()
                va = loss_fn(va_hat, Xva).item()
            print(f"[AE ] ep {ep:4d} | train_mse={tr:.6e} | val_mse={va:.6e}")

    return model.eval(), tpts.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()


@torch.no_grad()
def fae_reconstruct(
    model: FAEVanilla,
    X: torch.Tensor,
    Bp: torch.Tensor,
    Br: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    device = next(model.parameters()).device
    X = X.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)
    Xhat, H, _ = model(X, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu()


def train_fae_dynamics_head(
    model: FAEVanilla,
    Xn_train: torch.Tensor,
    Bp: torch.Tensor,
    cfg: FaeCfg,
) -> LatentDynamicsMLP:
    """
    Train nonlinear one-step latent transition on encoded FAE states.
    """
    set_seed(cfg.seed + 17)
    device = next(model.parameters()).device
    Bp = Bp.to(device)

    model.eval()
    with torch.no_grad():
        H = model.encode_from_curve(Xn_train.to(device).float(), Bp)

    H_in = H[:-1]
    H_out = H[1:]

    dyn = LatentDynamicsMLP(n_rep=cfg.n_rep, hidden=(64, 64)).to(device)
    init_small_weights(dyn, cfg.init_weight_sd)

    opt = optim.Adam(dyn.parameters(), lr=cfg.lr_dyn, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.dyn_epochs + 1):
        dyn.train()
        perm = torch.randperm(H_in.shape[0], device=device)

        for i in range(0, H_in.shape[0], cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            hin = H_in[idx]
            hout = H_out[idx]

            if cfg.dyn_teacher_forcing_noise > 0:
                hin = hin + cfg.dyn_teacher_forcing_noise * torch.randn_like(hin)

            pred = dyn(hin)
            loss = loss_fn(pred, hout)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            dyn.eval()
            with torch.no_grad():
                pred = dyn(H_in)
                tr = loss_fn(pred, H_out).item()
            print(f"[DYN] ep {ep:4d} | one_step_latent_mse={tr:.6e}")

    return dyn.eval()


@torch.no_grad()
def fae_nonlinear_forecast(
    model: FAEVanilla,
    dyn: LatentDynamicsMLP,
    Xn_train: torch.Tensor,
    steps: int,
    Bp: torch.Tensor,
    Br: torch.Tensor,
) -> torch.Tensor:
    device = next(model.parameters()).device
    Bp = Bp.to(device)
    Br = Br.to(device)

    H = model.encode_from_curve(Xn_train.to(device).float(), Bp)
    h = H[-1:].clone()

    preds = []
    for _ in range(steps):
        h = dyn(h)
        xhat = model.decode_from_rep(h, Br)
        preds.append(xhat.squeeze(0).detach().cpu())

    return torch.stack(preds, dim=0)


# ============================================================
# 7) One run
# ============================================================

def run_one_rep(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    rep: int,
    data_seed_base: int,
    verbose: bool = False,
) -> Dict[str, float]:
    data_seed = data_seed_base + rep
    sim = _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=data_seed)

    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    # ---------- FPCA reconstruction ----------
    X_fpca_recon_train, _, _ = fpca_reconstruct_uncentered_from_trainfit(
        X_train=Xn_train,
        X_eval=Xn_train,
        K=run_cfg.fpca_K,
    )

    # ---------- FAE reconstruction ----------
    old_log = fae_cfg.log_every
    if not verbose:
        fae_cfg.log_every = 0

    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_noisy_train(Xn_train, tpts, fae_cfg)
    Bp_dev = Bp_fae.to(fae_cfg.device)
    Br_dev = Br_fae.to(fae_cfg.device)

    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, Bp_dev, Br_dev)

    # ---------- Forecast ----------
    pca = PCA(n_components=run_cfg.fpca_K, svd_solver="full")
    scores = torch.tensor(pca.fit_transform(Xn_train.numpy()), dtype=torch.float32)

    Hf = kernel_forecast(
        scores,
        steps=run_cfg.horizon,
        bandwidth=1.0,
    )

    X_fpca_fore = torch.tensor(
        pca.inverse_transform(Hf.numpy()),
        dtype=torch.float32
    )

    with torch.no_grad():
        _, H_train = fae_reconstruct(model_fae, Xn_train, Bp_dev, Br_dev)

    H_train = H_train.to(fae_cfg.device)

    Hf = kernel_forecast(
        H_train,
        steps=run_cfg.horizon,
        bandwidth=1.0,
    )

    # decode
    X_fae_fore = model_fae.decode_from_rep(Hf.to(fae_cfg.device), Br_dev).cpu()

    fae_cfg.log_every = old_log

    # ---------- Metrics ----------
    # Reconstruction
    fpca_recon_rel_clean = rel_mse(X_fpca_recon_train, Xc_train)
    fpca_recon_rel_noisy = rel_mse(X_fpca_recon_train, Xn_train)
    fae_recon_rel_clean = rel_mse(X_fae_recon_train, Xc_train)
    fae_recon_rel_noisy = rel_mse(X_fae_recon_train, Xn_train)

    # Forecast
    fpca_fore_rel_clean = rel_mse(X_fpca_fore, Xc_test)
    fpca_fore_rel_noisy = rel_mse(X_fpca_fore, Xn_test)
    fae_fore_rel_clean = rel_mse(X_fae_fore, Xc_test)
    fae_fore_rel_noisy = rel_mse(X_fae_fore, Xn_test)

    # Final AE train/val MSE
    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=fae_cfg.split_rate,
        random_state=fae_cfg.seed,
        shuffle=True
    )
    Xtr = Xn_train[idx_tr].float()
    Xva = Xn_train[idx_va].float()

    model_fae.eval()
    with torch.no_grad():
        xtr_hat, _, _ = model_fae(Xtr.to(fae_cfg.device), Bp_dev, Br_dev)
        xva_hat, _, _ = model_fae(Xva.to(fae_cfg.device), Bp_dev, Br_dev)

    row = {
        "rep": rep,
        "data_seed": data_seed,

        "fpca_recon_relMSE_vs_clean": fpca_recon_rel_clean,
        "fpca_recon_relMSE_vs_noisy": fpca_recon_rel_noisy,
        "fae_recon_relMSE_vs_clean": fae_recon_rel_clean,
        "fae_recon_relMSE_vs_noisy": fae_recon_rel_noisy,

        "fae_train_mse_final_noisy": _mse(xtr_hat.detach().cpu(), Xtr),
        "fae_val_mse_final_noisy": _mse(xva_hat.detach().cpu(), Xva),

        "fpca_fore_relMSE_vs_clean": fpca_fore_rel_clean,
        "fpca_fore_relMSE_vs_noisy": fpca_fore_rel_noisy,
        "fae_fore_relMSE_vs_clean": fae_fore_rel_clean,
        "fae_fore_relMSE_vs_noisy": fae_fore_rel_noisy,
    }

    row.update({
        "map_mode": "nonlinear",
        "gen_basis_type": sim_cfg.gen_basis_type,
        "gen_n_basis": sim_cfg.gen_n_basis,
        "T": sim_cfg.T,
        "P": sim_cfg.P,
        "d": sim_cfg.d,
        "meas_noise_sd": sim_cfg.meas_noise_sd,
        "fpca_K": run_cfg.fpca_K,
        "horizon": run_cfg.horizon,
        "fae_n_rep": fae_cfg.n_rep,
        "fae_n_basis_project": fae_cfg.n_basis_project,
        "fae_n_basis_revert": fae_cfg.n_basis_revert,
        "latent_nl_scale": sim_cfg.latent_nl_scale,
        "latent_hetero_scale": sim_cfg.latent_hetero_scale,
    })
    return row


# ============================================================
# 8) Many runs
# ============================================================

def run_sim_fpca_fae_many(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 10,
    data_seed_base: int = 30000,
    verbose: bool = False,
) -> pd.DataFrame:
    fixed = build_fixed_decoder(sim_cfg)

    rows: List[Dict[str, float]] = []
    for rep in range(1, n_reps + 1):
        print(f"=== repetition {rep}/{n_reps} ===")
        row = run_one_rep(
            fixed=fixed,
            sim_cfg=sim_cfg,
            run_cfg=run_cfg,
            fae_cfg=fae_cfg,
            rep=rep,
            data_seed_base=data_seed_base,
            verbose=verbose,
        )
        rows.append(row)

        print(
            f"recon(clean): FPCA={row['fpca_recon_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_recon_relMSE_vs_clean']:.4f}"
        )
        print(
            f"fore(clean) : FPCA={row['fpca_fore_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_fore_relMSE_vs_clean']:.4f}"
        )
        print()

    df = pd.DataFrame(rows)
    return df


# ============================================================
# 9) Example
# ============================================================

if __name__ == "__main__":
    sim_cfg = SimCfg(
        seed=123,
        T=500,
        P=100,
        d=5,
        Sigma_scale=0.08,
        target_rho=0.80,
        gen_basis_type="bspline",
        gen_n_basis=12,
        gen_bspline_degree=3,
        map_hidden=(64, 64),
        map_activation="relu",
        map_weight_sd=0.8,
        meas_noise_sd=0.25,      # smaller than before
        latent_nl_scale=0.25,    # NEW
        latent_hetero_scale=0.15 # NEW
    )

    run_cfg = RunCfg(
        horizon=10,              # longer horizon helps expose nonlinear forecast gains
        fpca_K=5,
        fpca_var_ridge=1e-6,
    )

    fae_cfg = FaeCfg(
        seed=743,
        device="cpu",
        n_basis_project=30,
        n_basis_revert=30,
        basis_type_project="bspline",
        basis_type_revert="bspline",
        bspline_degree=3,
        n_rep=8,                 # slightly richer latent space than 5
        init_weight_sd=0.3,
        ae_epochs=1200,
        dyn_epochs=800,
        batch_size=32,
        lr_ae=3e-4,
        lr_dyn=3e-4,
        weight_decay=1e-4,
        split_rate=0.85,
        log_every=200,
        lamb=1e-4,
        dyn_teacher_forcing_noise=0.01,
    )

    df = run_sim_fpca_fae_many(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        data_seed_base=30000,
        verbose=False,
    )

    print("\n========== averages over repetitions ==========")
    avg_cols = [
        "fpca_recon_relMSE_vs_clean",
        "fae_recon_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_clean",
        "fae_fore_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_noisy",
        "fae_fore_relMSE_vs_noisy",
    ]
    print(df[avg_cols].mean())

    print("\n========== full table ==========")
    print(df.head(10))

=== repetition 1/10 ===


KeyboardInterrupt: 